# Notebook 14: Day 11 — Lithium plating ablation (partially reversible)

**Purpose**: ROADMAP v0.3 Item 1 — test whether activating PyBaMM lithium plating submodel changes X5-A's cell-internal Δt(Q80) sign distribution on the Chen2020/M50 cell. Independent of MJ1 comparison; sign-coincidence vs MJ1 is reported descriptively but not used as pass/fail.

**Configuration**: X5-A baseline (Chen2020 + V_init=2.82V + IDAKLU + 5-phase experiment) with one change: `options["lithium plating"] = "partially reversible"`.

**Mode choice**: partially reversible (NOT irreversible as initial ROADMAP draft specified). Rationale: the irreversible model has known physical-realism issues (PyBaMM Discussion #3447 by maintainer); partially reversible is recommended. ROADMAP will be updated post-Day 11.

**Acceptance criteria** (per ROADMAP):
- (a) Plating does not change X5-A sign on most cases → plating not key switch → next: Day 12 OCP slope scan
- (b) Plating flips sign on most cases → plating IS key switch → next: cross-parameter-set replication
- (c) Plating flips sign in some (DC, AC, f) regions → identify activation boundary → highest-value PhD-level outcome

**Plating parameters added** (Chen2020 lacks them by default):
- `Lithium plating kinetic rate constant [m.s-1]`: 1e-9 (PyBaMM example default)
- `Lithium plating transfer coefficient`: 0.5 (PyBaMM example default)
- `Dead lithium decay constant [s-1]`: 1e-4 (PyBaMM example default for partially reversible)

**Caveat**: above plating parameters are PyBaMM example defaults, not Chen2020-specific fitted values. Day 11 result is "Chen2020/M50 cell with default plating parameters" — sensitivity to these parameters is a v0.3 follow-up if Day 11 outcome (b) or (c).

In [1]:
# Day 11 Cell 1 — Environment probe + Chen2020 plating-parameter coverage
import pybamm
print(f"PyBaMM version: {pybamm.__version__}")
print()

# Step 1: Verify all three plating option strings build the model
print("=" * 70)
print("Step 1: Plating option strings — does PyBaMM accept them?")
print("=" * 70)

for cand in ["none", "irreversible", "reversible", "partially reversible"]:
    try:
        m = pybamm.lithium_ion.DFN(options={"thermal": "lumped", "lithium plating": cand})
        print(f"  '{cand}': ✓ accepted")
    except Exception as e:
        print(f"  '{cand}': ✗ {type(e).__name__}: {str(e)[:80]}")

# Step 2: Inspect Chen2020 parameter coverage for plating params
print()
print("=" * 70)
print("Step 2: Chen2020 — does it have plating parameters?")
print("=" * 70)

pv = pybamm.ParameterValues("Chen2020")
plating_keys = [
    "Lithium plating kinetic rate constant [m.s-1]",
    "Lithium plating transfer coefficient",
    "Dead lithium decay constant [s-1]",
    "Initial plated lithium concentration [mol.m-3]",
    "Typical plated lithium concentration [mol.m-3]",
    "Lithium metal partial molar volume [m3.mol-1]",
]
for k in plating_keys:
    val = pv.get(k, "NOT IN Chen2020")
    print(f"  {k:<55}: {val}")

# Step 3: Build the model and try parameterizing — what does PyBaMM say is missing?
print()
print("=" * 70)
print("Step 3: Build DFN+plating w/ Chen2020 — does parameterization complete?")
print("=" * 70)

for mode in ["irreversible", "partially reversible"]:
    print(f"\n  --- Trying mode='{mode}' ---")
    try:
        m = pybamm.lithium_ion.DFN(options={"thermal": "lumped", "lithium plating": mode})
        pv_test = pybamm.ParameterValues("Chen2020")
        pv_test.process_model(m)
        print(f"  ✓ '{mode}': parameterization complete WITHOUT extra params")
    except Exception as e:
        msg = str(e)
        # Extract the missing parameter name
        if "needed" in msg or "not provided" in msg or "KeyError" in str(type(e).__name__):
            # Show first 300 chars of error
            print(f"  ✗ '{mode}': {type(e).__name__}")
            print(f"    {msg[:400]}")
        else:
            print(f"  ✗ '{mode}': {type(e).__name__}: {msg[:200]}")

PyBaMM version: 26.3.1

Step 1: Plating option strings — does PyBaMM accept them?
  'none': ✓ accepted
  'irreversible': ✓ accepted
  'reversible': ✓ accepted
  'partially reversible': ✓ accepted

Step 2: Chen2020 — does it have plating parameters?
  Lithium plating kinetic rate constant [m.s-1]          : NOT IN Chen2020
  Lithium plating transfer coefficient                   : NOT IN Chen2020
  Dead lithium decay constant [s-1]                      : NOT IN Chen2020
  Initial plated lithium concentration [mol.m-3]         : NOT IN Chen2020
  Typical plated lithium concentration [mol.m-3]         : NOT IN Chen2020
  Lithium metal partial molar volume [m3.mol-1]          : NOT IN Chen2020

Step 3: Build DFN+plating w/ Chen2020 — does parameterization complete?

  --- Trying mode='irreversible' ---
  ✗ 'irreversible': KeyError
    "Parameter 'Exchange-current density for plating [A.m-2]' not found. 'Exchange-current density for plating [A.m-2]' not found. Best matches are ['SEI reactio

In [2]:
# Day 11 Cell 2 — Add plating parameters to Chen2020 + attempt model build
import pybamm
from pybamm import constants

# === OKane2020 exchange-current density functions ===
# Source: O'Kane et al. 2020 J. Electrochem. Soc. 167:090540
# (Verified against PyBaMM 26.3.1 stable docs; functional form preserved)

def plating_exchange_current_density_OKane2020(c_e, c_Li, T):
    """Exchange-current density for Li plating reaction [A.m-2]."""
    k_plating = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k_plating * c_e

def stripping_exchange_current_density_OKane2020(c_e, c_Li, T):
    """Exchange-current density for Li stripping reaction [A.m-2]."""
    k_plating = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k_plating * c_Li

# === Build parameter_values with full plating spec ===
pv = pybamm.ParameterValues("Chen2020")

plating_params = {
    # Kinetic + thermodynamic scalars
    "Lithium plating kinetic rate constant [m.s-1]": 1e-9,
    "Lithium plating transfer coefficient": 0.5,
    "Lithium metal partial molar volume [m3.mol-1]": 1.3e-5,
    "Initial plated lithium concentration [mol.m-3]": 0.0,
    "Typical plated lithium concentration [mol.m-3]": 1000.0,
    # Exchange-current density functions (OKane2020 form)
    "Exchange-current density for plating [A.m-2]":   plating_exchange_current_density_OKane2020,
    "Exchange-current density for stripping [A.m-2]": stripping_exchange_current_density_OKane2020,
    # Partially reversible additional
    "Dead lithium decay constant [s-1]": 1e-4,
}

try:
    pv.update(plating_params, check_already_exists=False)
    print(f"✓ Added {len(plating_params)} plating parameters to Chen2020")
except TypeError:
    # Fallback: direct assignment if check_already_exists keyword doesn't exist
    for k, v in plating_params.items():
        pv[k] = v
    print(f"✓ Added {len(plating_params)} plating parameters via direct assignment")

# === Try to build model + parameterize ===
print()
print("=" * 70)
print("Attempting: DFN + 'partially reversible' plating + Chen2020 + plating params")
print("=" * 70)

try:
    model = pybamm.lithium_ion.DFN(options={
        "thermal": "lumped",
        "lithium plating": "partially reversible",
    })
    pv.process_model(model)
    print(f"\n✓ SUCCESS — model fully parameterized")
    print(f"  Model name        : {model.name}")
    print(f"  Differential eqs  : {len(model.rhs)}")
    print(f"  Algebraic eqs     : {len(model.algebraic)}")
    print(f"  Boundary conds    : {len(model.boundary_conditions)}")
    print(f"\n→ Ready for V_rest sanity check (Cell 3)")
except Exception as e:
    print(f"\n✗ STILL MISSING SOMETHING: {type(e).__name__}")
    print(f"  Full message:")
    print(f"  {str(e)[:600]}")
    print(f"\n  → Tell me what's missing and I'll add it in Cell 2 v2")

✓ Added 8 plating parameters to Chen2020

Attempting: DFN + 'partially reversible' plating + Chen2020 + plating params

✗ STILL MISSING SOMETHING: KeyError
  Full message:
  "Parameter 'Dead lithium decay rate [s-1]' not found. 'Dead lithium decay rate [s-1]' not found. Best matches are ['Dead lithium decay constant [s-1]', 'Lithium plating kinetic rate constant [m.s-1]', 'Typical plated lithium concentration [mol.m-3]']"

  → Tell me what's missing and I'll add it in Cell 2 v2


In [3]:
# Day 11 Cell 2.1 — Add the final missing function parameter

def dead_lithium_decay_rate_constant(L_sei):
    """
    Dead lithium decay rate [s-1] — SEI-independent constant form.

    Returns the scalar 'Dead lithium decay constant [s-1]' regardless of L_sei.
    This is appropriate when SEI growth is not activated (our case).
    Physical interpretation: dead lithium decays at a fixed rate determined
    by spontaneous chemical degradation, decoupled from SEI thickness coupling.
    """
    return pybamm.Parameter("Dead lithium decay constant [s-1]")

# Add the missing function to pv (pv from previous cell still in memory)
try:
    pv.update({"Dead lithium decay rate [s-1]": dead_lithium_decay_rate_constant},
              check_already_exists=False)
    print("✓ Added 'Dead lithium decay rate [s-1]' function")
except TypeError:
    pv["Dead lithium decay rate [s-1]"] = dead_lithium_decay_rate_constant
    print("✓ Added via direct assignment")

# Re-attempt build
print()
print("=" * 70)
print("Re-attempting model build")
print("=" * 70)

try:
    model = pybamm.lithium_ion.DFN(options={
        "thermal": "lumped",
        "lithium plating": "partially reversible",
    })
    pv.process_model(model)
    print(f"\n✓ SUCCESS — model fully parameterized")
    print(f"  Model name        : {model.name}")
    print(f"  Differential eqs  : {len(model.rhs)}")
    print(f"  Algebraic eqs     : {len(model.algebraic)}")
    print(f"  Boundary conds    : {len(model.boundary_conditions)}")
    print(f"\n→ Ready for V_rest sanity check (Cell 3)")
except Exception as e:
    print(f"\n✗ STILL MISSING: {type(e).__name__}")
    print(f"  {str(e)[:600]}")

✓ Added 'Dead lithium decay rate [s-1]' function

Re-attempting model build

✓ SUCCESS — model fully parameterized
  Model name        : Doyle-Fuller-Newman model
  Differential eqs  : 8
  Algebraic eqs     : 3
  Boundary conds    : 7

→ Ready for V_rest sanity check (Cell 3)


In [4]:
# Day 11 Cell 3 — V_rest sanity check (does plating mode preserve X5-A V_init protocol?)
import numpy as np

# Step 1: Set initial SOC to V_init=2.82V (X5-A trick)
pv.set_initial_state(0.01688)
print("Step 1: set_initial_state(0.01688) applied")

# Step 2: 1-second rest experiment
exp_rest = pybamm.Experiment(["Rest for 1 second"])

# Step 3: Build sim, solve
sim = pybamm.Simulation(model, parameter_values=pv, experiment=exp_rest)
print(f"Step 2: solver auto-selected: {type(sim.solver).__name__}")

print("\nStep 3: Solving 1s rest...")
sol = sim.solve()
print(f"  ✓ Solved")

# Step 4: Discover plating-related variables (we don't know exact names in v26.3.1)
print("\nStep 4: Available plating/dead-Li/SEI variables in solution:")
plating_vars = []
for var_name in sol.all_models[0].variables.keys():
    if any(k in var_name.lower() for k in ["plating", "plated", "dead lith", "stripping"]):
        plating_vars.append(var_name)
        print(f"  - {var_name}")

# Step 5: Voltage check
print("\nStep 5: Voltage at rest")
t = sol["Time [s]"].entries
V = sol["Voltage [V]"].entries
print(f"  V at t=0   : {V[0]:.4f} V  (X5-A target: 2.8200 V)")
print(f"  V at t=1s  : {V[-1]:.4f} V")
print(f"  V drift    : {V[-1] - V[0]:+.6f} V  (should be ≈ 0 at rest)")

v_init_target = 2.82
v_init_tol = 0.05
if abs(V[0] - v_init_target) < v_init_tol:
    print(f"  ✓ V_init within ±{v_init_tol}V of target — X5-A protocol fix transfers to plating mode")
else:
    print(f"  ⚠ V_init off by {abs(V[0] - v_init_target):.4f}V — investigate before proceeding")

# Step 6: Plating activity at rest (should be ~0)
print("\nStep 6: Plating activity during rest")
candidate_loss_vars = [v for v in plating_vars if "loss" in v.lower() and "capacity" in v.lower()]
candidate_conc_vars = [v for v in plating_vars if "concentration" in v.lower()]

for var in candidate_loss_vars + candidate_conc_vars[:4]:  # first few
    try:
        arr = sol[var].entries
        if arr.ndim == 1:
            v0, v1 = arr[0], arr[-1]
        elif arr.ndim == 2:
            v0, v1 = arr[:, 0].mean(), arr[:, -1].mean()  # x-average
        else:
            continue
        delta = v1 - v0
        flag = " ✓ (no plating at rest)" if abs(delta) < 1e-10 else f" ⚠ ({delta:+.3e} change)"
        print(f"  {var:<70}: t0={v0:.3e}, t=1s={v1:.3e}{flag}")
    except Exception as e:
        print(f"  {var}: read error ({type(e).__name__})")

Step 1: set_initial_state(0.01688) applied
Step 2: solver auto-selected: IDAKLUSolver

Step 3: Solving 1s rest...
  ✓ Solved

Step 4: Available plating/dead-Li/SEI variables in solution:
  - Negative lithium plating concentration [mol.m-3]
  - X-averaged negative lithium plating concentration [mol.m-3]
  - Volume-averaged negative lithium plating concentration [mol.m-3]
  - Negative dead lithium concentration [mol.m-3]
  - X-averaged negative dead lithium concentration [mol.m-3]
  - Volume-averaged negative dead lithium concentration [mol.m-3]
  - Negative lithium plating thickness [m]
  - X-averaged negative lithium plating thickness [m]
  - Volume-averaged negative lithium plating thickness [m]
  - Negative dead lithium thickness [m]
  - X-averaged negative dead lithium thickness [m]
  - Volume-averaged negative dead lithium thickness [m]
  - Loss of lithium to negative lithium plating [mol]
  - Loss of capacity to negative lithium plating [A.h]
  - Positive lithium plating concentra

TypeError: Cannot convert symbol of type '<class 'pybamm.expression_tree.variable.Variable'>' to CasADi. Symbols must all be 'linear algebra' at this stage.

In [5]:
# Day 11 Cell 3.1 — Fresh build path (avoid re-processing already-processed model)
import pybamm
from pybamm import constants
import numpy as np

# === Redefine plating functions (in case kernel was restarted) ===
def plating_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_e

def stripping_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_Li

def dead_lithium_decay_rate_constant(L_sei):
    return pybamm.Parameter("Dead lithium decay constant [s-1]")

# === Fresh parameter_values (don't reuse pv from Cell 2 — it's been process_model'd) ===
pv_fresh = pybamm.ParameterValues("Chen2020")
pv_fresh.update({
    "Lithium plating kinetic rate constant [m.s-1]": 1e-9,
    "Lithium plating transfer coefficient": 0.5,
    "Lithium metal partial molar volume [m3.mol-1]": 1.3e-5,
    "Initial plated lithium concentration [mol.m-3]": 0.0,
    "Typical plated lithium concentration [mol.m-3]": 1000.0,
    "Exchange-current density for plating [A.m-2]":   plating_exchange_current_density_OKane2020,
    "Exchange-current density for stripping [A.m-2]": stripping_exchange_current_density_OKane2020,
    "Dead lithium decay constant [s-1]": 1e-4,
    "Dead lithium decay rate [s-1]": dead_lithium_decay_rate_constant,
}, check_already_exists=False)

# Apply set_initial_state BEFORE Simulation processes the model
pv_fresh.set_initial_state(0.01688)
print("✓ V_init=2.82V applied to fresh pv")

# === Fresh model (don't reuse model from Cell 2) ===
model_fresh = pybamm.lithium_ion.DFN(options={
    "thermal": "lumped",
    "lithium plating": "partially reversible",
})

# === Solve 1s rest ===
exp_rest = pybamm.Experiment(["Rest for 1 second"])
sim = pybamm.Simulation(model_fresh, parameter_values=pv_fresh, experiment=exp_rest)

print(f"\nSolving 1s rest...")
sol = sim.solve()
print(f"  ✓ Solved (solver: {type(sim.solver).__name__})")
print(f"  t_end = {sol['Time [s]'].entries[-1]:.3f}s, n_steps = {len(sol['Time [s]'].entries)}")

# === Try voltage access ===
print()
print("=" * 70)
print("Voltage check")
print("=" * 70)

V_value = None
V_name = None
for name in ["Voltage [V]", "Terminal voltage [V]", "X-averaged voltage [V]",
             "Battery voltage [V]", "Discharge voltage [V]"]:
    try:
        v = sol[name].entries
        if v.ndim == 1 and len(v) > 0:
            V_value = v
            V_name = name
            print(f"  ✓ '{name}': shape={v.shape}, V[0]={v[0]:.4f}V, V[-1]={v[-1]:.4f}V")
            break
        elif v.ndim == 2:
            V_value = v[:, -1].mean() if v.shape[1] else None
            V_name = name
            print(f"  ✓ '{name}' (2D): shape={v.shape}, mean V[-1]={v[:, -1].mean():.4f}V")
            break
    except Exception as e:
        print(f"  ✗ '{name}': {type(e).__name__}: {str(e)[:120]}")

if V_value is None:
    print("\n  ⚠ All voltage names failed — listing all available variables:")
    for var_name in list(sol.all_models[0].variables.keys())[:30]:
        if "voltage" in var_name.lower() or "potential" in var_name.lower():
            print(f"    candidate: {var_name}")

# === Sanity: V_init target check (only if V_value found) ===
if V_value is not None:
    if V_value.ndim == 1:
        v0 = V_value[0]
    else:
        v0 = float(V_value[:, 0].mean()) if V_value.ndim == 2 else float(V_value)
    print(f"\nV_init target: 2.82V")
    print(f"V[0]:          {v0:.4f}V")
    print(f"deviation:     {v0 - 2.82:+.4f}V")
    if abs(v0 - 2.82) < 0.05:
        print(f"  ✓ Within ±0.05V tolerance — X5-A protocol fix transfers to plating mode")
    elif abs(v0 - 2.82) < 0.20:
        print(f"  ⚠ Off by {abs(v0 - 2.82):.3f}V — investigate but not blocking")
    else:
        print(f"  ✗ Off by {abs(v0 - 2.82):.3f}V — STOP, debug before proceeding")

✓ V_init=2.82V applied to fresh pv

Solving 1s rest...
  ✓ Solved (solver: IDAKLUSolver)
  t_end = 1.000s, n_steps = 12

Voltage check
  ✓ 'Voltage [V]': shape=(12,), V[0]=2.8206V, V[-1]=2.8206V

V_init target: 2.82V
V[0]:          2.8206V
deviation:     +0.0006V
  ✓ Within ±0.05V tolerance — X5-A protocol fix transfers to plating mode


In [9]:
# Day 11 Cell 4 v3 — mirror run_single_case_DFN_X5 + add plating physics
# Critical fixes from v2:
#   (1) Phase 0a/0b/1 → 1s pseudo-rest (set_initial_state handles V_init)
#   (2) sin phase explicitly shifted by t_phase2_start_s = 1.0s
#   (3) termination="4.2V" + direction="charge" (no Upper V cut-off override needed)
#   (4) No period/duration on CustomStepExplicit (let IDAKLU + termination handle)
#   (5) Q_net via scipy cumulative_trapezoid on concatenated phase2+phase3

import pybamm
from pybamm import constants
from scipy.integrate import cumulative_trapezoid
import numpy as np
import time as wallclock

SOC_INIT_TARGET = 0.01688  # bisected SOC for V_init=2.82V on Chen2020 DFN

# === Plating physics (OKane2020 form) ===
def plating_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_e

def stripping_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_Li

def dead_lithium_decay_rate_constant(L_sei):
    return pybamm.Parameter("Dead lithium decay constant [s-1]")

PLATING_PARAMS = {
    "Lithium plating kinetic rate constant [m.s-1]": 1e-9,
    "Lithium plating transfer coefficient": 0.5,
    "Lithium metal partial molar volume [m3.mol-1]": 1.3e-5,
    "Initial plated lithium concentration [mol.m-3]": 0.0,
    "Typical plated lithium concentration [mol.m-3]": 1000.0,
    "Exchange-current density for plating [A.m-2]":   plating_exchange_current_density_OKane2020,
    "Exchange-current density for stripping [A.m-2]": stripping_exchange_current_density_OKane2020,
    "Dead lithium decay constant [s-1]": 1e-4,
    "Dead lithium decay rate [s-1]": dead_lithium_decay_rate_constant,
}


def run_single_case_plating(
    I_DC_Crate, A_Crate, f_Hz,
    parameter_set="Chen2020", T_ambient_C=20.0, nominal_capacity_Ah=5.0,
    soc_init=SOC_INIT_TARGET, pseudo_rest_s=1.0,
    plating_mode="partially reversible",
    verbose=False,
):
    """X5-A protocol + plating activation."""
    if A_Crate is None or (isinstance(A_Crate, float) and np.isnan(A_Crate)):
        A_Crate = 0.0
    if f_Hz is None or (isinstance(f_Hz, float) and np.isnan(f_Hz)):
        f_Hz = 0.0

    I_DC_A = -abs(I_DC_Crate) * nominal_capacity_Ah
    A_A    = A_Crate * nominal_capacity_Ah
    kappa  = (A_Crate / abs(I_DC_Crate)) if abs(I_DC_Crate) > 1e-9 else 0.0

    case_id = (f"DC{abs(I_DC_Crate):.2f}C+AC{A_Crate:.2f}C_f{f_Hz:.5f}Hz_X5plating"
               if A_Crate >= 1e-9
               else f"DC{abs(I_DC_Crate):.2f}C_baseline_X5plating")

    t_phase2_start_s = pseudo_rest_s

    def dc_ac_current(variables):
        t_absolute = variables["Time [s]"]
        t_relative = t_absolute - t_phase2_start_s
        return I_DC_A + A_A * pybamm.sin(2 * np.pi * f_Hz * t_relative)

    dcac_step = pybamm.step.CustomStepExplicit(
        dc_ac_current, termination="4.2V", direction="charge",
        description=f"Phase 2: {case_id}",
    )

    experiment = pybamm.Experiment([
        f"Rest for {pseudo_rest_s} seconds",
        dcac_step,
        "Hold at 4.2V until C/68",
    ])

    model = pybamm.lithium_ion.DFN(options={
        "thermal": "lumped",
        "lithium plating": plating_mode,
    })
    pv = pybamm.ParameterValues(parameter_set)
    pv["Ambient temperature [K]"] = T_ambient_C + 273.15
    pv["Initial temperature [K]"] = T_ambient_C + 273.15
    pv.update(PLATING_PARAMS, check_already_exists=False)
    pv.set_initial_state(soc_init)

    sim = pybamm.Simulation(model, parameter_values=pv, experiment=experiment)

    t_start = wallclock.time()
    try:
        solution = sim.solve()
    except Exception as e:
        return {
            "case_id": case_id, "I_DC_Crate": abs(I_DC_Crate), "A_Crate": A_Crate, "f_Hz": f_Hz,
            "kappa": kappa, "wall_clock_s": wallclock.time() - t_start,
            "status": f"FAILED_SOLVE: {type(e).__name__}: {e}",
        }
    wall_clock_s = wallclock.time() - t_start

    n_cycles = len(solution.cycles)
    base_dict = {
        "case_id": case_id, "I_DC_Crate": abs(I_DC_Crate), "A_Crate": A_Crate, "f_Hz": f_Hz,
        "kappa": kappa, "wall_clock_s": wall_clock_s, "soc_init": soc_init,
    }

    if n_cycles < 2:
        return {**base_dict, "status": f"INFEASIBLE_EARLY: only {n_cycles} cycles"}
    if n_cycles < 3:
        phase2_partial = solution.cycles[1]
        return {
            **base_dict,
            "status": f"INFEASIBLE_MIN_V: only 2 cycles (no CV phase reached)",
            "CC_time_s_partial": phase2_partial.t[-1] - phase2_partial.t[0],
            "V_min_phase2": float(phase2_partial["Voltage [V]"].entries.min()),
        }

    phase2 = solution.cycles[1]
    phase3 = solution.cycles[2]
    cc_time_s = phase2.t[-1] - phase2.t[0]
    cv_time_s = phase3.t[-1] - phase3.t[0]
    total_time_s = cc_time_s + cv_time_s

    t_chg = np.concatenate([phase2.t, phase3.t])
    I_chg = np.concatenate([phase2["Current [A]"].entries, phase3["Current [A]"].entries])
    _, unique_idx = np.unique(t_chg, return_index=True)
    unique_idx_sorted = np.sort(unique_idx)
    t_chg = t_chg[unique_idx_sorted]
    I_chg = I_chg[unique_idx_sorted]
    t_chg = t_chg - t_chg[0]

    Q_net_As = -cumulative_trapezoid(I_chg, t_chg, initial=0)
    Q_net_mAh = Q_net_As / 3600.0 * 1000.0
    Q_final_mAh = float(Q_net_mAh[-1])

    T_max_charging = max(
        float(phase2["X-averaged cell temperature [K]"].entries.max() - 273.15),
        float(phase3["X-averaged cell temperature [K]"].entries.max() - 273.15),
    )
    V_init = float(phase2["Voltage [V]"].entries[0])

    # Plating-specific outputs
    plating_loss_p2 = float(phase2["Loss of capacity to negative lithium plating [A.h]"].entries[-1] -
                            phase2["Loss of capacity to negative lithium plating [A.h]"].entries[0])
    plating_loss_p3 = float(phase3["Loss of capacity to negative lithium plating [A.h]"].entries[-1] -
                            phase3["Loss of capacity to negative lithium plating [A.h]"].entries[0])

    if verbose:
        print(f"  [{case_id}] V_init={V_init:.4f}V CC={cc_time_s/60:.2f}min "
              f"T_max={T_max_charging:.2f}°C Q={Q_final_mAh:.0f}mAh "
              f"plating_loss={(plating_loss_p2+plating_loss_p3)*1000:.1f}mAh "
              f"wall={wall_clock_s:.2f}s")

    return {
        **base_dict,
        "V_init_phase2": V_init,
        "CC_time_s": cc_time_s, "CV_time_s": cv_time_s, "total_time_s": total_time_s,
        "CC_time_min": cc_time_s/60, "CV_time_min": cv_time_s/60, "total_time_min": total_time_s/60,
        "Q_net_final_mAh": Q_final_mAh,
        "Q_net_trajectory": Q_net_mAh.tolist(),
        "t_chg": t_chg.tolist(),
        "I_chg": I_chg.tolist(),
        "T_max_charging_C": T_max_charging,
        "plating_loss_p2_Ah": plating_loss_p2,
        "plating_loss_p3_Ah": plating_loss_p3,
        "plating_loss_total_mAh": (plating_loss_p2 + plating_loss_p3) * 1000.0,
        "status": "ok",
    }


# === Single case test: 0.2+0.8C 10τ (κ=4) ===
print("=" * 70)
print("Single case test: 0.2+0.8C 10τ with X5-A protocol + partially reversible plating")
print("=" * 70)
result = run_single_case_plating(I_DC_Crate=0.2, A_Crate=0.8, f_Hz=0.001430, verbose=True)
print()
for k, v in result.items():
    if k in ["Q_net_trajectory", "t_chg", "I_chg"]:
        print(f"  {k:<25}: array of len {len(v)}")
    else:
        print(f"  {k:<25}: {v}")

Single case test: 0.2+0.8C 10τ with X5-A protocol + partially reversible plating
  [DC0.20C+AC0.80C_f0.00143Hz_X5plating] V_init=2.8873V CC=230.21min T_max=27.15°C Q=5043mAh plating_loss=19.7mAh wall=0.75s

  case_id                  : DC0.20C+AC0.80C_f0.00143Hz_X5plating
  I_DC_Crate               : 0.2
  A_Crate                  : 0.8
  f_Hz                     : 0.00143
  kappa                    : 4.0
  wall_clock_s             : 0.7473649978637695
  soc_init                 : 0.01688
  V_init_phase2            : 2.8873348476726957
  CC_time_s                : 13812.302819168845
  CV_time_s                : 4646.532890074366
  total_time_s             : 18458.835709243212
  CC_time_min              : 230.2050469861474
  CV_time_min              : 77.44221483457277
  total_time_min           : 307.6472618207202
  Q_net_final_mAh          : 5042.8999812178345
  Q_net_trajectory         : array of len 1415
  t_chg                    : array of len 1415
  I_chg                    : arr

In [10]:
# Quick sanity: X5-A baseline (no plating) V_init under identical set_initial_state(0.01688) + 1s rest
import pybamm
import numpy as np

pv_x5a = pybamm.ParameterValues("Chen2020")
pv_x5a["Ambient temperature [K]"] = 20.0 + 273.15
pv_x5a["Initial temperature [K]"] = 20.0 + 273.15
pv_x5a.set_initial_state(0.01688)

model_x5a = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
sim_x5a = pybamm.Simulation(model_x5a, parameter_values=pv_x5a, experiment=pybamm.Experiment(["Rest for 1 second"]))
sol_x5a = sim_x5a.solve()
v_x5a = sol_x5a["Voltage [V]"].entries
print(f"X5-A (no plating) V_init at SOC=0.01688: {v_x5a[0]:.4f} V")
print(f"  X5-A drift in 1s rest: {v_x5a[-1] - v_x5a[0]:+.4f} V")
print(f"  Day 11 (with plating) V_init was: 2.8873 V")
print(f"  Difference: {2.8873 - v_x5a[0]:+.4f} V")

X5-A (no plating) V_init at SOC=0.01688: 2.8206 V
  X5-A drift in 1s rest: -0.0000 V
  Day 11 (with plating) V_init was: 2.8873 V
  Difference: +0.0667 V


In [11]:
# Bisect SOC for V_init = 2.82V under plating mode
import pybamm
from pybamm import constants
import numpy as np

# === Re-define plating physics (idempotent) ===
def plating_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_e

def stripping_exchange_current_density_OKane2020(c_e, c_Li, T):
    k = pybamm.Parameter("Lithium plating kinetic rate constant [m.s-1]")
    return constants.F * k * c_Li

def dead_lithium_decay_rate_constant(L_sei):
    return pybamm.Parameter("Dead lithium decay constant [s-1]")

PLATING_PARAMS = {
    "Lithium plating kinetic rate constant [m.s-1]": 1e-9,
    "Lithium plating transfer coefficient": 0.5,
    "Lithium metal partial molar volume [m3.mol-1]": 1.3e-5,
    "Initial plated lithium concentration [mol.m-3]": 0.0,
    "Typical plated lithium concentration [mol.m-3]": 1000.0,
    "Exchange-current density for plating [A.m-2]":   plating_exchange_current_density_OKane2020,
    "Exchange-current density for stripping [A.m-2]": stripping_exchange_current_density_OKane2020,
    "Dead lithium decay constant [s-1]": 1e-4,
    "Dead lithium decay rate [s-1]": dead_lithium_decay_rate_constant,
}

def get_v_init_at_soc(soc, plating_mode="partially reversible"):
    """Run a 1s rest at given SOC, return V_init."""
    pv = pybamm.ParameterValues("Chen2020")
    pv["Ambient temperature [K]"] = 20.0 + 273.15
    pv["Initial temperature [K]"] = 20.0 + 273.15
    pv.update(PLATING_PARAMS, check_already_exists=False)
    pv.set_initial_state(soc)
    
    model = pybamm.lithium_ion.DFN(options={"thermal": "lumped", "lithium plating": plating_mode})
    sim = pybamm.Simulation(model, parameter_values=pv,
                            experiment=pybamm.Experiment(["Rest for 1 second"]))
    sol = sim.solve()
    return float(sol["Voltage [V]"].entries[0])

# === Bisect SOC to give V_init = 2.82V under plating ===
print("=" * 70)
print("Bisecting SOC for V_init = 2.82V under partially reversible plating")
print("=" * 70)

V_TARGET = 2.82
TOL = 0.001  # 1 mV tolerance

# Initial bounds: V_init at SOC=0.01688 was 2.8873 → need lower SOC for lower V
# X5-A baseline: SOC=0.01688 → V=2.82. Plating shifts V up ~67mV at same SOC
# So plating SOC for V=2.82 is LOWER than 0.01688
# Bracket: lo=0.001, hi=0.01688

lo, hi = 0.001, 0.01688
v_lo = get_v_init_at_soc(lo)
v_hi = get_v_init_at_soc(hi)
print(f"  bracket: SOC={lo:.5f} → V={v_lo:.4f}V  |  SOC={hi:.5f} → V={v_hi:.4f}V")

if not (v_lo < V_TARGET < v_hi):
    print(f"  ⚠ V_TARGET={V_TARGET}V not in bracket — extending bounds")
    if V_TARGET <= v_lo:
        # Need even lower SOC
        lo = 0.0005
        v_lo = get_v_init_at_soc(lo)
        print(f"    extended lo: SOC={lo:.5f} → V={v_lo:.4f}V")

# Bisect
print(f"\n  Bisecting (target {V_TARGET}V, tol {TOL*1000:.1f}mV):")
for it in range(20):
    mid = 0.5 * (lo + hi)
    v_mid = get_v_init_at_soc(mid)
    print(f"    iter {it+1:>2}: SOC={mid:.6f} → V={v_mid:.4f}V")
    if abs(v_mid - V_TARGET) < TOL:
        print(f"\n  ✓ Converged: SOC={mid:.6f} → V_init={v_mid:.4f}V")
        SOC_INIT_PLATING = mid
        break
    if v_mid > V_TARGET:
        hi = mid
    else:
        lo = mid
else:
    print(f"\n  ⚠ Max iterations reached. Best: SOC={mid:.6f} → V_init={v_mid:.4f}V")
    SOC_INIT_PLATING = mid

print(f"\n*** SOC_INIT_PLATING = {SOC_INIT_PLATING:.6f} ***")
print(f"*** Use this in run_single_case_plating(soc_init=SOC_INIT_PLATING) ***")

Bisecting SOC for V_init = 2.82V under partially reversible plating
  bracket: SOC=0.00100 → V=2.5245V  |  SOC=0.01688 → V=2.8206V

  Bisecting (target 2.82V, tol 1.0mV):
    iter  1: SOC=0.008940 → V=2.6922V
    iter  2: SOC=0.012910 → V=2.7607V
    iter  3: SOC=0.014895 → V=2.7917V
    iter  4: SOC=0.015887 → V=2.8064V
    iter  5: SOC=0.016384 → V=2.8136V
    iter  6: SOC=0.016632 → V=2.8171V
    iter  7: SOC=0.016756 → V=2.8189V
    iter  8: SOC=0.016818 → V=2.8198V

  ✓ Converged: SOC=0.016818 → V_init=2.8198V

*** SOC_INIT_PLATING = 0.016818 ***
*** Use this in run_single_case_plating(soc_init=SOC_INIT_PLATING) ***


In [12]:
# Day 11 Cell 5 — 24-case DC+AC batch + 6 DC baseline + Δt(Q80) computation
# Mirror X5-A batch structure; reuse run_single_case_plating from Cell 4 v3

import pandas as pd
import numpy as np
from pathlib import Path

# === Load existing 4-way table to get exact (DC, AC, f_Hz) tuples ===
repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
df_4way = pd.read_csv(repo / "data" / "results_day8_x5_4way_dt_Q80_v2.csv")

# Dedup (row 24 is duplicate of row 7)
df_cases = df_4way.drop_duplicates(subset='condition', keep='first').reset_index(drop=True)
print(f"Loaded 24 unique DC+AC cases:")
print(df_cases[['condition', 'DC', 'AC', 'f_Hz', 'kappa', 'tau_label']].to_string())

# === DC baseline cases (6 unique DC C-rates needed for Δt(Q80) reference) ===
unique_DCs = sorted(df_cases['DC'].unique())
print(f"\n6 DC baseline cases needed at C-rates: {unique_DCs}")

# === Run all 24 DC+AC cases ===
print("\n" + "=" * 70)
print("Running 24 DC+AC cases (X5-A protocol + partially reversible plating)")
print("=" * 70)

dcac_results = []
for i, row in df_cases.iterrows():
    print(f"\n[{i+1}/24] {row['condition']} (κ={row['kappa']:.2f}, f={row['f_Hz']*1000:.2f}mHz)...")
    res = run_single_case_plating(
        I_DC_Crate=row['DC'], A_Crate=row['AC'], f_Hz=row['f_Hz'],
        verbose=True,
    )
    res['condition'] = row['condition']
    res['tau_label'] = row['tau_label']
    res['exp_dt_Q80_min'] = row['dt_exp']  # carry forward MJ1 exp value
    res['X5A_dt_Q80_min'] = row['dt_X5A']  # carry X5-A baseline for direct comparison
    dcac_results.append(res)

# === Run 6 DC baseline cases (A=0, f=0) ===
print("\n" + "=" * 70)
print("Running 6 DC-only baseline cases for Δt(Q80) reference")
print("=" * 70)

dc_results = {}
for dc in unique_DCs:
    print(f"\n[DC={dc}C only]...")
    res = run_single_case_plating(
        I_DC_Crate=dc, A_Crate=0.0, f_Hz=0.0,
        verbose=True,
    )
    if res['status'] == 'ok':
        # Find t(Q80) on the DC-only Q_net trajectory
        Q_arr = np.array(res['Q_net_trajectory'])
        t_arr = np.array(res['t_chg'])
        Q80 = 4101.0
        above = Q_arr >= Q80
        if above.any():
            idx = np.argmax(above)
            if idx > 0:
                t_lo, t_hi = t_arr[idx-1], t_arr[idx]
                Q_lo, Q_hi = Q_arr[idx-1], Q_arr[idx]
                t_Q80 = t_lo + (Q80 - Q_lo) / (Q_hi - Q_lo) * (t_hi - t_lo)
            else:
                t_Q80 = t_arr[0]
            dc_results[dc] = {
                't_Q80_s': t_Q80,
                't_Q80_min': t_Q80 / 60,
                'CC_time_min': res['CC_time_min'],
                'Q_net_final_mAh': res['Q_net_final_mAh'],
                'plating_loss_mAh': res['plating_loss_total_mAh'],
            }
            print(f"  → DC={dc}C ref t(Q80) = {t_Q80/60:.3f} min, plating loss = {res['plating_loss_total_mAh']:.1f} mAh")
        else:
            dc_results[dc] = None
            print(f"  ⚠ DC={dc}C did not reach Q80")
    else:
        dc_results[dc] = None
        print(f"  ✗ {res['status']}")

# === Compute Δt(Q80) for each DC+AC case ===
print("\n" + "=" * 70)
print("Computing Δt(Q80) = t_DC_ref - t_DCAC for each case")
print("=" * 70)

batch_summary = []
for res in dcac_results:
    if res['status'] != 'ok':
        batch_summary.append({
            'condition': res['condition'], 'kappa': res['kappa'],
            'tau_label': res['tau_label'],
            'DC': res['I_DC_Crate'], 'AC': res['A_Crate'], 'f_Hz': res['f_Hz'],
            't_Q80_DCAC_min': None, 't_Q80_DC_ref_min': None,
            'dt_Q80_plating_min': None,
            'dt_Q80_X5A_min': res['X5A_dt_Q80_min'],
            'dt_Q80_exp_min': res['exp_dt_Q80_min'],
            'CC_time_min': None, 'plating_loss_mAh': None,
            'status': res['status'],
        })
        continue

    # Find t(Q80) for this DC+AC case
    Q_arr = np.array(res['Q_net_trajectory'])
    t_arr = np.array(res['t_chg'])
    Q80 = 4101.0
    above = Q_arr >= Q80
    t_Q80_DCAC = None
    if above.any():
        idx = np.argmax(above)
        if idx > 0:
            t_lo, t_hi = t_arr[idx-1], t_arr[idx]
            Q_lo, Q_hi = Q_arr[idx-1], Q_arr[idx]
            t_Q80_DCAC = t_lo + (Q80 - Q_lo) / (Q_hi - Q_lo) * (t_hi - t_lo)
        else:
            t_Q80_DCAC = t_arr[0]

    # DC reference at matching C-rate
    dc_ref = dc_results.get(res['I_DC_Crate'])
    t_Q80_DC = dc_ref['t_Q80_s'] if dc_ref else None
    dt_Q80 = (t_Q80_DC - t_Q80_DCAC) / 60.0 if (t_Q80_DC is not None and t_Q80_DCAC is not None) else None

    batch_summary.append({
        'condition': res['condition'], 'kappa': res['kappa'],
        'tau_label': res['tau_label'],
        'DC': res['I_DC_Crate'], 'AC': res['A_Crate'], 'f_Hz': res['f_Hz'],
        't_Q80_DCAC_min': t_Q80_DCAC/60 if t_Q80_DCAC else None,
        't_Q80_DC_ref_min': dc_ref['t_Q80_min'] if dc_ref else None,
        'dt_Q80_plating_min': dt_Q80,
        'dt_Q80_X5A_min': res['X5A_dt_Q80_min'],
        'dt_Q80_exp_min': res['exp_dt_Q80_min'],
        'CC_time_min': res['CC_time_min'],
        'plating_loss_mAh': res['plating_loss_total_mAh'],
        'status': res['status'],
    })

df_batch = pd.DataFrame(batch_summary)
print(df_batch.to_string())

# === Save to CSV ===
out_path = repo / "data" / "results_day11_plating_dt_Q80.csv"
df_batch.to_csv(out_path, index=False)
print(f"\n✓ Saved: {out_path}")
print(f"  shape: {df_batch.shape}")

# === Quick sign-coincidence summary ===
print("\n" + "=" * 70)
print("Quick sign-coincidence summary (threshold |Δt|<0.10 min)")
print("=" * 70)

def sgn(x, thr=0.10):
    if pd.isna(x): return None
    return 0 if abs(x) < thr else (1 if x > 0 else -1)

mask_valid = df_batch['dt_Q80_plating_min'].notna() & df_batch['dt_Q80_exp_min'].notna()
df_valid = df_batch[mask_valid].copy()

s_plating = df_valid['dt_Q80_plating_min'].apply(sgn)
s_X5A     = df_valid['dt_Q80_X5A_min'].apply(sgn)
s_exp     = df_valid['dt_Q80_exp_min'].apply(sgn)

n_total = len(df_valid)
agree_plating_vs_exp = (s_plating == s_exp).sum()
agree_X5A_vs_exp     = (s_X5A == s_exp).sum()
agree_plating_vs_X5A = (s_plating == s_X5A).sum()
flips_plating_vs_X5A = (s_plating != s_X5A).sum()

print(f"  Total valid cases: {n_total}")
print(f"  Sign-coincidence vs MJ1 exp:")
print(f"    Day 11 plating:  {agree_plating_vs_exp}/{n_total} ({100*agree_plating_vs_exp/n_total:.1f}%)")
print(f"    X5-A baseline:   {agree_X5A_vs_exp}/{n_total} ({100*agree_X5A_vs_exp/n_total:.1f}%)")
print(f"  Day 11 vs X5-A internal comparison:")
print(f"    Sign agreement:  {agree_plating_vs_X5A}/{n_total}")
print(f"    Sign FLIP:       {flips_plating_vs_X5A}/{n_total} cases where plating flips X5-A's Δt(Q80) sign")

# === ROADMAP acceptance criterion classification ===
print(f"\n  ROADMAP acceptance criterion (vs Day 11 spec a/b/c):")
n_flip = flips_plating_vs_X5A
if n_flip <= 4:  # ≤ 4 out of 24
    print(f"  → Outcome (a): plating no major effect ({n_flip}/24 sign flips vs X5-A)")
    print(f"    → Day 12: proceed with OCP slope scan as planned")
elif n_flip >= 16:  # ≥ 16 out of 24
    print(f"  → Outcome (b): plating IS the key switch ({n_flip}/24 sign flips vs X5-A)")
    print(f"    → Day 12: cross-parameter-set replication (O'Kane2022, Marquis2019, Mohtat2020)")
else:
    print(f"  → Outcome (c): plating selectively flips ({n_flip}/24)")
    print(f"    → Day 12: identify activation boundary in (DC, AC, f) space")

Loaded 24 unique DC+AC cases:
         condition   DC   AC     f_Hz     kappa tau_label
0      0.1+0.9C 1τ  0.1  0.9  0.01430  9.000000        1τ
1      0.1+0.2C 1τ  0.1  0.2  0.01430  2.000000        1τ
2      0.2+0.3C 1τ  0.2  0.3  0.01430  1.500000        1τ
3     0.2+0.3C 10τ  0.2  0.3  0.00143  1.500000       10τ
4   0.2+0.3C 34.8τ  0.2  0.3  0.00041  1.500000     34.8τ
5    0.2+0.8C 0.1τ  0.2  0.8  0.14300  4.000000      0.1τ
6      0.2+0.8C 1τ  0.2  0.8  0.01430  4.000000        1τ
7     0.2+0.8C 10τ  0.2  0.8  0.00143  4.000000       10τ
8    0.3+0.7C 0.1τ  0.3  0.7  0.14300  2.333333      0.1τ
9      0.3+0.7C 1τ  0.3  0.7  0.01430  2.333333        1τ
10    0.3+0.7C 10τ  0.3  0.7  0.00143  2.333333       10τ
11   0.3+0.4C 0.1τ  0.3  0.4  0.14300  1.333333      0.1τ
12     0.3+0.4C 1τ  0.3  0.4  0.01430  1.333333        1τ
13    0.3+0.4C 10τ  0.3  0.4  0.00143  1.333333       10τ
14   0.4+0.6C 0.5τ  0.4  0.6  0.02860  1.500000      0.5τ
15  0.4+0.5C 1.67τ  0.4  0.5  0.00860  1.2

In [13]:
# Verify row 10 stability: rerun + compare neighbor cases
from copy import deepcopy

# Re-run row 10 with verbose
print("=" * 70)
print("Re-run row 10: 0.3+0.7C 10τ (κ=2.33, f=1.43mHz)")
print("=" * 70)
res_row10_repeat = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.7, f_Hz=0.001430, verbose=True)
print(f"\nRepeat dt_Q80: ...")
# Compute dt_Q80 manually from this run + DC=0.3 baseline
Q_arr = np.array(res_row10_repeat['Q_net_trajectory'])
t_arr = np.array(res_row10_repeat['t_chg'])
above = Q_arr >= 4101.0
idx = np.argmax(above)
t_lo, t_hi = t_arr[idx-1], t_arr[idx]
Q_lo, Q_hi = Q_arr[idx-1], Q_arr[idx]
t_Q80_repeat = t_lo + (4101.0 - Q_lo) / (Q_hi - Q_lo) * (t_hi - t_lo)
dt_repeat = (dc_results[0.3]['t_Q80_s'] - t_Q80_repeat) / 60
print(f"  Original run: dt_Q80 = -0.134 min")
print(f"  Repeat run:   dt_Q80 = {dt_repeat:+.3f} min")
print(f"  Difference:   {abs(dt_repeat - (-0.134)):.4f} min (should be < 0.001 if reproducible)")

# Compare neighbor κ at 10τ to see κ pattern
print(f"\nκ pattern at 10τ (f=1.43mHz):")
print(f"  κ=1.33 (0.3+0.4C 10τ):  X5A=+0.77, plating=+0.82 (Δ +0.05)")
print(f"  κ=1.50 (0.4+0.6C 10τ):  X5A=+2.28, plating=+2.41 (Δ +0.13)")
print(f"  κ=1.50 (0.2+0.3C 10τ):  X5A=-5.30, plating=-5.30 (Δ +0.00)")
print(f"  κ=2.33 (0.3+0.7C 10τ):  X5A=+7.27, plating=-0.13 (Δ -7.40) ← outlier")
print(f"  κ=4.00 (0.2+0.8C 10τ):  X5A=+9.33, plating=+9.41 (Δ +0.08)")
print(f"\nPattern: row 10 (κ=2.33) is outlier vs κ=1.33/1.50/4.00 neighbors at same f.")

# Also compare neighbor f at κ=2.33
print(f"\nf pattern at κ=2.33 (DC=0.3, AC=0.7):")
print(f"  f=143mHz (0.1τ):    X5A=+2.49, plating=+2.27 (Δ -0.22)")
print(f"  f=14.3mHz (1τ):     X5A=+3.59, plating=+2.95 (Δ -0.64)")
print(f"  f=1.43mHz (10τ):    X5A=+7.27, plating=-0.13 (Δ -7.40) ← outlier")
print(f"\nPattern: at κ=2.33, plating effect grows monotonically with decreasing f, then jumps at 10τ.")

Re-run row 10: 0.3+0.7C 10τ (κ=2.33, f=1.43mHz)
  [DC0.30C+AC0.70C_f0.00143Hz_X5plating] V_init=2.9095V CC=159.34min T_max=26.49°C Q=5043mAh plating_loss=19.3mAh wall=0.65s

Repeat dt_Q80: ...
  Original run: dt_Q80 = -0.134 min
  Repeat run:   dt_Q80 = -0.134 min
  Difference:   0.0002 min (should be < 0.001 if reproducible)

κ pattern at 10τ (f=1.43mHz):
  κ=1.33 (0.3+0.4C 10τ):  X5A=+0.77, plating=+0.82 (Δ +0.05)
  κ=1.50 (0.4+0.6C 10τ):  X5A=+2.28, plating=+2.41 (Δ +0.13)
  κ=1.50 (0.2+0.3C 10τ):  X5A=-5.30, plating=-5.30 (Δ +0.00)
  κ=2.33 (0.3+0.7C 10τ):  X5A=+7.27, plating=-0.13 (Δ -7.40) ← outlier
  κ=4.00 (0.2+0.8C 10τ):  X5A=+9.33, plating=+9.41 (Δ +0.08)

Pattern: row 10 (κ=2.33) is outlier vs κ=1.33/1.50/4.00 neighbors at same f.

f pattern at κ=2.33 (DC=0.3, AC=0.7):
  f=143mHz (0.1τ):    X5A=+2.49, plating=+2.27 (Δ -0.22)
  f=14.3mHz (1τ):     X5A=+3.59, plating=+2.95 (Δ -0.64)
  f=1.43mHz (10τ):    X5A=+7.27, plating=-0.13 (Δ -7.40) ← outlier

Pattern: at κ=2.33, plating

In [14]:
# Stress test row 10: compute Δt(Q*) at Q40/Q60/Q80 to test metric-artifact vs physics-effect hypothesis
import numpy as np

print("=" * 70)
print("Row 10 Δt(Q*) at multiple Q* targets")
print("=" * 70)

# Re-extract row 10 trajectory (already in memory from Cell 5 batch run, but let's be explicit)
res_row10 = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.7, f_Hz=0.001430)
res_dc03  = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.0, f_Hz=0.0)

def first_passage_time(Q_traj, t_traj, Q_target):
    Q_arr = np.array(Q_traj)
    t_arr = np.array(t_traj)
    above = Q_arr >= Q_target
    if not above.any():
        return None
    idx = np.argmax(above)
    if idx == 0:
        return float(t_arr[0])
    Q_lo, Q_hi = Q_arr[idx-1], Q_arr[idx]
    t_lo, t_hi = t_arr[idx-1], t_arr[idx]
    return float(t_lo + (Q_target - Q_lo) / (Q_hi - Q_lo) * (t_hi - t_lo))

# Q40 / Q60 / Q80 / Q90 first-passage
Q_targets = {
    "Q20 (1025 mAh)":  1025.0,
    "Q40 (2050 mAh)":  2050.0,
    "Q60 (3076 mAh)":  3076.0,
    "Q70 (3588 mAh)":  3588.0,
    "Q80 (4101 mAh)":  4101.0,
    "Q90 (4613 mAh)":  4613.0,
}

print(f"{'Q* target':<20} {'t_DCAC [min]':>14} {'t_DC [min]':>14} {'Δt [min]':>10} {'X5-A Δt expected':>20}")
print("-" * 80)
# Note: we only have X5-A Δt at Q80 = +7.27. Others would need running X5-A separately.
# But the pattern (artifact vs physics) is testable from plating run alone.
for label, Q in Q_targets.items():
    t_DCAC = first_passage_time(res_row10['Q_net_trajectory'], res_row10['t_chg'], Q)
    t_DC   = first_passage_time(res_dc03['Q_net_trajectory'],  res_dc03['t_chg'],  Q)
    if t_DCAC is None or t_DC is None:
        print(f"{label:<20} {'—':>14} {'—':>14} {'—':>10}")
        continue
    dt_min = (t_DC - t_DCAC) / 60
    print(f"{label:<20} {t_DCAC/60:>14.3f} {t_DC/60:>14.3f} {dt_min:>+10.3f}")

print()
print("Interpretation:")
print("  - If Δt is similar across Q40/Q60/Q70/Q80 → plating effect is ROBUST (real physics)")
print("  - If Δt is large only at Q80 → metric artifact (first-passage AC half-cycle sensitivity)")
print("  - X5-A baseline at row 10 is +7.27 min for Q80; we don't have its full Q* curve")
print("    but if plating's Q* curve is non-monotonic, X5-A's likely is too — both metric-driven")

Row 10 Δt(Q*) at multiple Q* targets
Q* target              t_DCAC [min]     t_DC [min]   Δt [min]     X5-A Δt expected
--------------------------------------------------------------------------------
Q20 (1025 mAh)               44.195         41.000     -3.195
Q40 (2050 mAh)               89.056         82.000     -7.056
Q60 (3076 mAh)              125.924        123.040     -2.884
Q70 (3588 mAh)              148.373        143.520     -4.853
Q80 (4101 mAh)              164.174        164.040     -0.134
Q90 (4613 mAh)              177.645        184.520     +6.875

Interpretation:
  - If Δt is similar across Q40/Q60/Q70/Q80 → plating effect is ROBUST (real physics)
  - If Δt is large only at Q80 → metric artifact (first-passage AC half-cycle sensitivity)
  - X5-A baseline at row 10 is +7.27 min for Q80; we don't have its full Q* curve
    but if plating's Q* curve is non-monotonic, X5-A's likely is too — both metric-driven


In [15]:
# Day 11 stress test cell — Δt(Q*) curve diagnostic on 3 representative cases
# Goal: verify whether Q80 first-passage instability is row 10 unique or systematic across X5-A baseline

import numpy as np

# Pick 3 cases with different X5-A Δt(Q80) magnitudes:
# - row 7: 0.2+0.8C 10τ — large positive X5-A (+9.33), should be stable
# - row 3: 0.2+0.3C 10τ — large negative X5-A (-5.30), should be stable  
# - row 13: 0.3+0.4C 10τ — small positive X5-A (+0.77), borderline

stress_cases = [
    (0.2, 0.8, 0.001430, "0.2+0.8C 10τ", "row 7"),
    (0.2, 0.3, 0.001430, "0.2+0.3C 10τ", "row 3"),
    (0.3, 0.4, 0.001430, "0.3+0.4C 10τ", "row 13"),
]

Q_targets = [1025, 2050, 3076, 3588, 4101, 4613]
Q_labels  = ["Q20", "Q40", "Q60", "Q70", "Q80", "Q90"]

def fpt(Q_traj, t_traj, Q_target):
    Q = np.array(Q_traj); t = np.array(t_traj)
    above = Q >= Q_target
    if not above.any(): return None
    idx = np.argmax(above)
    if idx == 0: return float(t[0])
    return float(t[idx-1] + (Q_target - Q[idx-1]) / (Q[idx] - Q[idx-1]) * (t[idx] - t[idx-1]))

# Pre-compute DC baseline trajectories for each DC C-rate involved
dc_runs = {}
for DC in [0.2, 0.3]:
    if DC not in dc_runs:
        r = run_single_case_plating(I_DC_Crate=DC, A_Crate=0.0, f_Hz=0.0)
        dc_runs[DC] = r

# Run each stress case and tabulate
print(f"{'case':<25} ", end="")
for lbl in Q_labels:
    print(f"{lbl:>9}", end="")
print(f"  {'Δt range':>10}  {'oscillation?':>14}")
print("-" * 110)

for DC, AC, f_Hz, label, row_id in stress_cases:
    res = run_single_case_plating(I_DC_Crate=DC, A_Crate=AC, f_Hz=f_Hz)
    dc_ref = dc_runs[DC]
    
    dt_curve = []
    for Q in Q_targets:
        t_DCAC = fpt(res['Q_net_trajectory'], res['t_chg'], Q)
        t_DC   = fpt(dc_ref['Q_net_trajectory'], dc_ref['t_chg'], Q)
        if t_DCAC is None or t_DC is None:
            dt_curve.append(None)
        else:
            dt_curve.append((t_DC - t_DCAC) / 60)
    
    valid = [x for x in dt_curve if x is not None]
    dt_range = max(valid) - min(valid) if valid else 0
    is_oscillating = "⚠ YES" if dt_range > 5 else "✓ stable"
    
    print(f"{label + ' (' + row_id + ')':<25} ", end="")
    for v in dt_curve:
        print(f"{v:>+9.2f}" if v is not None else f"{'—':>9}", end="")
    print(f"  {dt_range:>+10.2f}  {is_oscillating:>14}")

# Compare to row 10 reference
print()
print("Row 10 reference (from previous test):")
print("  0.3+0.7C 10τ:  Δt = -3.20, -7.06, -2.88, -4.85, -0.13, +6.88  (range 14 min)  ⚠ OSCILLATING")

case                            Q20      Q40      Q60      Q70      Q80      Q90    Δt range    oscillation?
--------------------------------------------------------------------------------------------------------------
0.2+0.8C 10τ (row 7)          -5.90    -3.36    -0.97    -4.11    +9.41   +26.75      +32.64           ⚠ YES
0.2+0.3C 10τ (row 3)          -4.37    -2.49    -0.64    -3.06    -5.30    -1.70       +4.66        ✓ stable
0.3+0.4C 10τ (row 13)         -2.61    -0.09    -2.34    -3.90    +0.82    +7.72      +11.62           ⚠ YES

Row 10 reference (from previous test):
  0.3+0.7C 10τ:  Δt = -3.20, -7.06, -2.88, -4.85, -0.13, +6.88  (range 14 min)  ⚠ OSCILLATING


In [16]:
# Combined stress test:
# (Step 1) Row 10 X5-A no-plating Δt(Q*) curve
# (Step 2) 3-case Day 11 plating Δt(Q*) check on different X5-A magnitudes
# (Step 3) Plating - X5A diff at row 10 across Q*

import pybamm
import numpy as np
import time as wallclock
from scipy.integrate import cumulative_trapezoid

# === X5-A no-plating runner (mirror run_single_case_DFN_X5 from notebook 10) ===
SOC_INIT_TARGET = 0.01688

def run_single_case_X5A_noplating(I_DC_Crate, A_Crate, f_Hz,
                                   T_ambient_C=20.0, nominal_capacity_Ah=5.0,
                                   soc_init=SOC_INIT_TARGET, pseudo_rest_s=1.0):
    if A_Crate is None or (isinstance(A_Crate, float) and np.isnan(A_Crate)):
        A_Crate = 0.0
    if f_Hz is None or (isinstance(f_Hz, float) and np.isnan(f_Hz)):
        f_Hz = 0.0
    
    I_DC_A = -abs(I_DC_Crate) * nominal_capacity_Ah
    A_A    = A_Crate * nominal_capacity_Ah
    t_phase2_start_s = pseudo_rest_s
    
    def dc_ac_current(variables):
        t_absolute = variables["Time [s]"]
        t_relative = t_absolute - t_phase2_start_s
        return I_DC_A + A_A * pybamm.sin(2 * np.pi * f_Hz * t_relative)
    
    dcac_step = pybamm.step.CustomStepExplicit(
        dc_ac_current, termination="4.2V", direction="charge",
    )
    experiment = pybamm.Experiment([
        f"Rest for {pseudo_rest_s} seconds",
        dcac_step,
        "Hold at 4.2V until C/68",
    ])
    
    model = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
    pv = pybamm.ParameterValues("Chen2020")
    pv["Ambient temperature [K]"] = T_ambient_C + 273.15
    pv["Initial temperature [K]"] = T_ambient_C + 273.15
    pv.set_initial_state(soc_init)
    
    sim = pybamm.Simulation(model, parameter_values=pv, experiment=experiment)
    solution = sim.solve()
    
    if len(solution.cycles) < 3:
        return {"status": f"INFEASIBLE: only {len(solution.cycles)} cycles"}
    
    phase2 = solution.cycles[1]; phase3 = solution.cycles[2]
    t_chg = np.concatenate([phase2.t, phase3.t])
    I_chg = np.concatenate([phase2["Current [A]"].entries, phase3["Current [A]"].entries])
    _, unique_idx = np.unique(t_chg, return_index=True)
    unique_idx_sorted = np.sort(unique_idx)
    t_chg = t_chg[unique_idx_sorted] - t_chg[unique_idx_sorted][0]
    I_chg = I_chg[unique_idx_sorted]
    Q_net_mAh = -cumulative_trapezoid(I_chg, t_chg, initial=0) / 3600.0 * 1000.0
    
    return {"status": "ok", "t_chg": t_chg.tolist(), "Q_net_trajectory": Q_net_mAh.tolist(),
            "CC_time_min": (phase2.t[-1] - phase2.t[0]) / 60}

def fpt(Q_traj, t_traj, Q_target):
    Q = np.array(Q_traj); t = np.array(t_traj)
    above = Q >= Q_target
    if not above.any(): return None
    idx = np.argmax(above)
    if idx == 0: return float(t[0])
    return float(t[idx-1] + (Q_target - Q[idx-1]) / (Q[idx] - Q[idx-1]) * (t[idx] - t[idx-1]))

Q_targets = [1025, 2050, 3076, 3588, 4101, 4613]
Q_labels  = ["Q20", "Q40", "Q60", "Q70", "Q80", "Q90"]

# === Step 1: Row 10 X5-A no-plating + DC=0.3 X5-A baseline ===
print("=" * 90)
print("Step 1: Row 10 (0.3+0.7C 10τ) under X5-A NO plating + DC=0.3 baseline")
print("=" * 90)

print("\nRunning X5-A no-plating, row 10 (0.3+0.7C 10τ)...")
t0 = wallclock.time()
res_row10_X5A = run_single_case_X5A_noplating(0.3, 0.7, 0.001430)
print(f"  Solved in {wallclock.time()-t0:.2f}s, status={res_row10_X5A['status']}")

print("\nRunning X5-A no-plating, DC=0.3 baseline...")
t0 = wallclock.time()
res_dc03_X5A = run_single_case_X5A_noplating(0.3, 0.0, 0.0)
print(f"  Solved in {wallclock.time()-t0:.2f}s, status={res_dc03_X5A['status']}")

# === Step 2: Tabulate Δt(Q*) curves for row 10 (X5-A no-plating + plating) ===
# Re-fetch row 10 plating result and DC=0.3 plating baseline
print("\nReusing row 10 plating + DC=0.3 plating baseline from earlier...")
res_row10_plating = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.7, f_Hz=0.001430)
res_dc03_plating  = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.0, f_Hz=0.0)

print("\n" + "=" * 90)
print("Row 10 Δt(Q*) curves: X5-A baseline vs Day 11 plating vs DIFFERENCE")
print("=" * 90)
print(f"{'Q*':<8} {'t_DCAC_X5A':>12} {'t_DC_X5A':>12} {'Δt_X5A':>10} | "
      f"{'t_DCAC_pl':>12} {'t_DC_pl':>12} {'Δt_pl':>10} | {'diff (pl-X5A)':>14}")
print("-" * 110)

x5a_dt_curve = []
plating_dt_curve = []
for Q, lbl in zip(Q_targets, Q_labels):
    t_DCAC_X5A  = fpt(res_row10_X5A['Q_net_trajectory'], res_row10_X5A['t_chg'], Q)
    t_DC_X5A    = fpt(res_dc03_X5A['Q_net_trajectory'],  res_dc03_X5A['t_chg'],  Q)
    t_DCAC_pl   = fpt(res_row10_plating['Q_net_trajectory'], res_row10_plating['t_chg'], Q)
    t_DC_pl     = fpt(res_dc03_plating['Q_net_trajectory'],  res_dc03_plating['t_chg'],  Q)
    
    dt_X5A = (t_DC_X5A - t_DCAC_X5A) / 60 if (t_DC_X5A and t_DCAC_X5A) else None
    dt_pl  = (t_DC_pl  - t_DCAC_pl)  / 60 if (t_DC_pl  and t_DCAC_pl)  else None
    diff = dt_pl - dt_X5A if (dt_pl is not None and dt_X5A is not None) else None
    
    x5a_dt_curve.append(dt_X5A)
    plating_dt_curve.append(dt_pl)
    
    s_dt_X5A = f"{dt_X5A:>+10.3f}" if dt_X5A is not None else f"{'—':>10}"
    s_dt_pl  = f"{dt_pl:>+10.3f}" if dt_pl is not None else f"{'—':>10}"
    s_diff   = f"{diff:>+14.3f}" if diff is not None else f"{'—':>14}"
    s_t_DCAC_X5A = f"{t_DCAC_X5A/60:>12.3f}" if t_DCAC_X5A else f"{'—':>12}"
    s_t_DC_X5A   = f"{t_DC_X5A/60:>12.3f}"   if t_DC_X5A   else f"{'—':>12}"
    s_t_DCAC_pl  = f"{t_DCAC_pl/60:>12.3f}"  if t_DCAC_pl  else f"{'—':>12}"
    s_t_DC_pl    = f"{t_DC_pl/60:>12.3f}"    if t_DC_pl    else f"{'—':>12}"
    
    print(f"{lbl:<8} {s_t_DCAC_X5A} {s_t_DC_X5A} {s_dt_X5A} | {s_t_DCAC_pl} {s_t_DC_pl} {s_dt_pl} | {s_diff}")

# === Step 3: Range analysis ===
print()
valid_X5A = [x for x in x5a_dt_curve if x is not None]
valid_pl  = [x for x in plating_dt_curve if x is not None]
range_X5A = max(valid_X5A) - min(valid_X5A) if valid_X5A else 0
range_pl  = max(valid_pl)  - min(valid_pl)  if valid_pl  else 0
print(f"X5-A baseline Δt(Q*) range: {range_X5A:.2f} min ({'⚠ OSCILLATING' if range_X5A > 5 else '✓ stable'})")
print(f"Day 11 plating Δt(Q*) range: {range_pl:.2f} min ({'⚠ OSCILLATING' if range_pl > 5 else '✓ stable'})")

# === Step 4: 3-case stress test under plating ===
print()
print("=" * 90)
print("Step 4: 3-case Δt(Q*) stress test under Day 11 plating")
print("=" * 90)

stress_cases = [
    (0.2, 0.8, 0.001430, "0.2+0.8C 10τ", "row 7",  "+9.33"),
    (0.2, 0.3, 0.001430, "0.2+0.3C 10τ", "row 3",  "-5.30"),
    (0.3, 0.4, 0.001430, "0.3+0.4C 10τ", "row 13", "+0.77"),
]

dc_plating_runs = {0.2: run_single_case_plating(I_DC_Crate=0.2, A_Crate=0.0, f_Hz=0.0),
                   0.3: res_dc03_plating}

print(f"{'case (X5A-Δt80)':<32} ", end="")
for lbl in Q_labels:
    print(f"{lbl:>9}", end="")
print(f"  {'Δt range':>10}  {'verdict':>14}")
print("-" * 130)

for DC, AC, f_Hz, label, row_id, x5a_dt80 in stress_cases:
    res = run_single_case_plating(I_DC_Crate=DC, A_Crate=AC, f_Hz=f_Hz)
    dc_ref = dc_plating_runs[DC]
    
    dt_curve = []
    for Q in Q_targets:
        t_DCAC = fpt(res['Q_net_trajectory'], res['t_chg'], Q)
        t_DC   = fpt(dc_ref['Q_net_trajectory'], dc_ref['t_chg'], Q)
        dt_curve.append((t_DC - t_DCAC) / 60 if (t_DC and t_DCAC) else None)
    valid = [x for x in dt_curve if x is not None]
    dt_range = max(valid) - min(valid) if valid else 0
    
    print(f"{label + ' (' + row_id + ', ' + x5a_dt80 + ')':<32} ", end="")
    for v in dt_curve:
        print(f"{v:>+9.2f}" if v is not None else f"{'—':>9}", end="")
    verdict = "⚠ OSCILLATING" if dt_range > 5 else "✓ stable"
    print(f"  {dt_range:>+10.2f}  {verdict:>14}")

Step 1: Row 10 (0.3+0.7C 10τ) under X5-A NO plating + DC=0.3 baseline

Running X5-A no-plating, row 10 (0.3+0.7C 10τ)...
  Solved in 0.89s, status=ok

Running X5-A no-plating, DC=0.3 baseline...
  Solved in 0.61s, status=ok

Reusing row 10 plating + DC=0.3 plating baseline from earlier...

Row 10 Δt(Q*) curves: X5-A baseline vs Day 11 plating vs DIFFERENCE
Q*         t_DCAC_X5A     t_DC_X5A     Δt_X5A |    t_DCAC_pl      t_DC_pl      Δt_pl |  diff (pl-X5A)
--------------------------------------------------------------------------------------------------------------
Q20            44.196       41.000     -3.196 |       44.195       41.000     -3.195 |         +0.001
Q40            89.053       82.000     -7.053 |       89.056       82.000     -7.056 |         -0.003
Q60           125.930      123.040     -2.890 |      125.924      123.040     -2.884 |         +0.006
Q70           148.382      143.520     -4.862 |      148.373      143.520     -4.853 |         +0.008
Q80           156.77

In [17]:
# Stress test phase 2: full 24-case Δt(Q*) range scan under both X5-A baseline AND Day 11 plating
# Goal: identify which cases are metric-stable vs metric-unstable
# Output: stability flag per case, supports deciding which Day 11 / v0.2 sign-coincidence numbers are robust

import pandas as pd
import numpy as np

# Load all 24 case definitions from 4-way table
df_cases = df_4way.drop_duplicates(subset='condition', keep='first').reset_index(drop=True)

# Q* range definition
Q_targets = [1025, 2050, 3076, 3588, 4101, 4613]  # Q20 / Q40 / Q60 / Q70 / Q80 / Q90
Q_labels = ["Q20", "Q40", "Q60", "Q70", "Q80", "Q90"]

# Pre-compute X5-A and plating DC baselines for each unique DC
print("Pre-computing 6 DC baselines under X5-A no-plating...")
dc_X5A = {}
for DC in sorted(df_cases['DC'].unique()):
    dc_X5A[DC] = run_single_case_X5A_noplating(DC, 0.0, 0.0)
    print(f"  DC={DC}C X5-A: status={dc_X5A[DC]['status']}")

# Plating DC baselines already computed in dc_results (from Cell 5) — fetch them
print("\nUsing existing plating DC baselines from earlier batch run...")

# === Per-case Δt(Q*) range and Q80-vs-curve-mean comparison ===
print("\n" + "=" * 110)
print("Per-case Δt(Q*) curve stability scan (X5-A baseline + Day 11 plating)")
print("=" * 110)

stability_summary = []

for i, row in df_cases.iterrows():
    DC = row['DC']; AC = row['AC']; f_Hz = row['f_Hz']
    cond = row['condition']
    
    # X5-A no-plating
    res_X5A = run_single_case_X5A_noplating(DC, AC, f_Hz)
    if res_X5A['status'] != 'ok':
        stability_summary.append({'condition': cond, 'kappa': row['kappa'], 'tau_label': row['tau_label'],
                                   'X5A_Q80': None, 'X5A_range': None, 'X5A_stable': False,
                                   'pl_Q80': None, 'pl_range': None, 'pl_stable': False})
        continue
    
    # Plating
    res_pl = run_single_case_plating(I_DC_Crate=DC, A_Crate=AC, f_Hz=f_Hz)
    
    # Compute Δt curves
    dc_ref_X5A = dc_X5A[DC]
    
    X5A_dt_curve = []
    pl_dt_curve = []
    for Q in Q_targets:
        t_DCAC_X5A = fpt(res_X5A['Q_net_trajectory'], res_X5A['t_chg'], Q)
        t_DC_X5A = fpt(dc_ref_X5A['Q_net_trajectory'], dc_ref_X5A['t_chg'], Q)
        X5A_dt_curve.append((t_DC_X5A - t_DCAC_X5A)/60 if (t_DC_X5A and t_DCAC_X5A) else None)
        
        # Plating: use dc_results[DC] from earlier Cell 5 batch run
        # (Need to reconstruct — for now, recompute since cheap)
        # Actually let's skip plating recompute since we have it: use res_pl which has trajectory
        # but we need DC-only plating reference for matching DC
        # Reuse from dc_runs / dc_results dict — let's just refetch
    
    # For plating, use existing dc_results from Cell 5
    if DC in dc_results and dc_results[DC] is not None:
        dc_ref_pl_t = np.array(dc_results[DC]).get('t_arr', None) if isinstance(dc_results[DC], dict) else None
        # Actually dc_results dict structure was different. Let's compute fresh.
        pass
    
    # Just recompute plating DC for each unique DC (cheap, only 6 of them)
    # For efficiency, cache:
    if not hasattr(run_single_case_plating, '_dc_cache'):
        run_single_case_plating._dc_cache = {}
    if DC not in run_single_case_plating._dc_cache:
        run_single_case_plating._dc_cache[DC] = run_single_case_plating(I_DC_Crate=DC, A_Crate=0.0, f_Hz=0.0)
    dc_ref_pl = run_single_case_plating._dc_cache[DC]
    
    for Q in Q_targets:
        t_DCAC_pl = fpt(res_pl['Q_net_trajectory'], res_pl['t_chg'], Q)
        t_DC_pl = fpt(dc_ref_pl['Q_net_trajectory'], dc_ref_pl['t_chg'], Q)
        pl_dt_curve.append((t_DC_pl - t_DCAC_pl)/60 if (t_DC_pl and t_DCAC_pl) else None)
    
    # Stability: range over Q* curve
    valid_X5A = [x for x in X5A_dt_curve if x is not None]
    valid_pl = [x for x in pl_dt_curve if x is not None]
    X5A_range = max(valid_X5A) - min(valid_X5A) if valid_X5A else None
    pl_range = max(valid_pl) - min(valid_pl) if valid_pl else None
    
    stability_summary.append({
        'condition': cond, 'kappa': row['kappa'], 'tau_label': row['tau_label'],
        'X5A_Q80': X5A_dt_curve[4],  # index 4 = Q80
        'X5A_range': X5A_range,
        'X5A_stable': X5A_range < 5.0 if X5A_range else False,
        'pl_Q80': pl_dt_curve[4],
        'pl_range': pl_range,
        'pl_stable': pl_range < 5.0 if pl_range else False,
    })
    print(f"  [{i+1:2}/24] {cond:<20} κ={row['kappa']:.2f} {row['tau_label']:<6}  "
          f"X5-A range={X5A_range:>5.2f}min ({'✓' if X5A_range and X5A_range<5 else '⚠'})  "
          f"plating range={pl_range:>5.2f}min ({'✓' if pl_range and pl_range<5 else '⚠'})")

df_stab = pd.DataFrame(stability_summary)

# === Summary ===
print("\n" + "=" * 110)
print("Stability summary")
print("=" * 110)
n_X5A_stable = df_stab['X5A_stable'].sum()
n_pl_stable = df_stab['pl_stable'].sum()
print(f"X5-A baseline:   {n_X5A_stable}/24 cases metric-stable (Δt(Q*) range < 5 min)")
print(f"Day 11 plating:  {n_pl_stable}/24 cases metric-stable")

print(f"\nUnstable cases (Δt(Q*) range > 5 min):")
unstable = df_stab[~df_stab['X5A_stable'] | ~df_stab['pl_stable']]
print(unstable[['condition', 'kappa', 'tau_label', 'X5A_range', 'pl_range']].to_string())

# Save to CSV for next steps
df_stab.to_csv(repo / "data" / "results_day11_metric_stability_scan.csv", index=False)
print(f"\n✓ Saved: data/results_day11_metric_stability_scan.csv")

Pre-computing 6 DC baselines under X5-A no-plating...
  DC=0.1C X5-A: status=ok
  DC=0.2C X5-A: status=ok
  DC=0.3C X5-A: status=ok
  DC=0.4C X5-A: status=ok
  DC=0.5C X5-A: status=ok
  DC=0.9C X5-A: status=ok

Using existing plating DC baselines from earlier batch run...

Per-case Δt(Q*) curve stability scan (X5-A baseline + Day 11 plating)


AttributeError: 'numpy.ndarray' object has no attribute 'get'

In [18]:
# Day 11 metric stability scan — clean rewrite
# Per-case Δt(Q*) range across Q20/40/60/70/80/90 under both X5-A baseline and Day 11 plating

import pandas as pd
import numpy as np

Q_targets = [1025, 2050, 3076, 3588, 4101, 4613]
Q_labels = ["Q20", "Q40", "Q60", "Q70", "Q80", "Q90"]

df_cases = df_4way.drop_duplicates(subset='condition', keep='first').reset_index(drop=True)

# === Pre-compute 6 DC baselines under X5-A no-plating ===
print("Pre-computing 6 DC baselines under X5-A no-plating...")
dc_X5A = {}
for DC in sorted(df_cases['DC'].unique()):
    dc_X5A[DC] = run_single_case_X5A_noplating(DC, 0.0, 0.0)
print(f"  ✓ Done.")

# === Pre-compute 6 DC baselines under Day 11 plating ===
print("\nPre-computing 6 DC baselines under Day 11 plating...")
dc_pl = {}
for DC in sorted(df_cases['DC'].unique()):
    dc_pl[DC] = run_single_case_plating(I_DC_Crate=DC, A_Crate=0.0, f_Hz=0.0)
print(f"  ✓ Done.")

# === Per-case Δt(Q*) curve scan ===
print("\n" + "=" * 130)
print("Per-case Δt(Q*) range scan (24 cases × 2 configurations)")
print("=" * 130)

stability_summary = []
for i, row in df_cases.iterrows():
    DC, AC, f_Hz = row['DC'], row['AC'], row['f_Hz']
    cond = row['condition']
    
    # X5-A no-plating
    res_X5A = run_single_case_X5A_noplating(DC, AC, f_Hz)
    # Day 11 plating
    res_pl = run_single_case_plating(I_DC_Crate=DC, A_Crate=AC, f_Hz=f_Hz)
    
    X5A_dt = []
    pl_dt = []
    for Q in Q_targets:
        # X5-A
        if res_X5A.get('status') == 'ok':
            t_DCAC = fpt(res_X5A['Q_net_trajectory'], res_X5A['t_chg'], Q)
            t_DC   = fpt(dc_X5A[DC]['Q_net_trajectory'], dc_X5A[DC]['t_chg'], Q)
            X5A_dt.append((t_DC - t_DCAC)/60 if (t_DC and t_DCAC) else None)
        else:
            X5A_dt.append(None)
        # Plating
        if res_pl.get('status') == 'ok':
            t_DCAC = fpt(res_pl['Q_net_trajectory'], res_pl['t_chg'], Q)
            t_DC   = fpt(dc_pl[DC]['Q_net_trajectory'], dc_pl[DC]['t_chg'], Q)
            pl_dt.append((t_DC - t_DCAC)/60 if (t_DC and t_DCAC) else None)
        else:
            pl_dt.append(None)
    
    valid_X5A = [x for x in X5A_dt if x is not None]
    valid_pl  = [x for x in pl_dt if x is not None]
    X5A_range = max(valid_X5A) - min(valid_X5A) if valid_X5A else None
    pl_range  = max(valid_pl)  - min(valid_pl)  if valid_pl  else None
    
    stability_summary.append({
        'condition': cond, 'kappa': row['kappa'], 'tau_label': row['tau_label'],
        'DC': DC, 'AC': AC, 'f_Hz': f_Hz,
        'X5A_Q80': X5A_dt[4], 'X5A_Q40': X5A_dt[1], 'X5A_Q90': X5A_dt[5],
        'X5A_range': X5A_range,
        'X5A_stable': (X5A_range is not None and X5A_range < 5.0),
        'pl_Q80': pl_dt[4], 'pl_Q40': pl_dt[1], 'pl_Q90': pl_dt[5],
        'pl_range': pl_range,
        'pl_stable': (pl_range is not None and pl_range < 5.0),
    })
    
    x5a_flag = '✓' if (X5A_range is not None and X5A_range < 5) else '⚠'
    pl_flag  = '✓' if (pl_range  is not None and pl_range  < 5) else '⚠'
    x5a_str  = f"{X5A_range:>5.2f}" if X5A_range is not None else "  N/A"
    pl_str   = f"{pl_range:>5.2f}"  if pl_range  is not None else "  N/A"
    print(f"  [{i+1:2}/24] {cond:<22} κ={row['kappa']:>5.2f} {row['tau_label']:<7}  "
          f"X5A range={x5a_str}min {x5a_flag}  pl range={pl_str}min {pl_flag}")

df_stab = pd.DataFrame(stability_summary)

# === Summary ===
print("\n" + "=" * 80)
print("Stability summary")
print("=" * 80)
n_X5A_stab = df_stab['X5A_stable'].sum()
n_pl_stab  = df_stab['pl_stable'].sum()
print(f"X5-A baseline:    {n_X5A_stab}/24 cases stable (Δt(Q*) range < 5 min)")
print(f"Day 11 plating:   {n_pl_stab}/24 cases stable")

print(f"\nCases unstable in EITHER config:")
unstable = df_stab[~df_stab['X5A_stable'] | ~df_stab['pl_stable']]
cols = ['condition', 'kappa', 'tau_label', 'X5A_range', 'pl_range']
print(unstable[cols].to_string(index=False))

# Cross-tab by tau_label
print(f"\nUnstable cases grouped by τ_label (frequency family):")
for tau in sorted(df_stab['tau_label'].unique()):
    sub = df_stab[df_stab['tau_label'] == tau]
    n_x5a_unst = (~sub['X5A_stable']).sum()
    n_pl_unst = (~sub['pl_stable']).sum()
    n_total = len(sub)
    print(f"  {tau:<8}: X5-A {n_x5a_unst}/{n_total} unstable, plating {n_pl_unst}/{n_total} unstable")

# Save
df_stab.to_csv(repo / "data" / "results_day11_metric_stability_scan.csv", index=False)
print(f"\n✓ Saved: data/results_day11_metric_stability_scan.csv")

Pre-computing 6 DC baselines under X5-A no-plating...
  ✓ Done.

Pre-computing 6 DC baselines under Day 11 plating...
  ✓ Done.

Per-case Δt(Q*) range scan (24 cases × 2 configurations)
  [ 1/24] 0.1+0.9C 1τ            κ= 9.00 1τ       X5A range=52.14min ⚠  pl range=50.39min ⚠
  [ 2/24] 0.1+0.2C 1τ            κ= 2.00 1τ       X5A range= 0.57min ✓  pl range= 0.54min ✓
  [ 3/24] 0.2+0.3C 1τ            κ= 1.50 1τ       X5A range= 2.43min ✓  pl range= 1.93min ✓
  [ 4/24] 0.2+0.3C 10τ           κ= 1.50 10τ      X5A range= 9.69min ⚠  pl range= 4.66min ✓
  [ 5/24] 0.2+0.3C 34.8τ         κ= 1.50 34.8τ    X5A range=14.66min ⚠  pl range=14.65min ⚠
  [ 6/24] 0.2+0.8C 0.1τ          κ= 4.00 0.1τ     X5A range=19.21min ⚠  pl range=19.02min ⚠
  [ 7/24] 0.2+0.8C 1τ            κ= 4.00 1τ       X5A range=22.01min ⚠  pl range=21.42min ⚠
  [ 8/24] 0.2+0.8C 10τ           κ= 4.00 10τ      X5A range=32.41min ⚠  pl range=32.64min ⚠
  [ 9/24] 0.3+0.7C 0.1τ          κ= 2.33 0.1τ     X5A range= 9.55min ⚠  pl ran

In [20]:
# Day 11 v3 metric — JES2-aligned curve metrics
# A_Δt = ∫Δt(Q) dQ over Q20-Q80 (CC phase only, with safety margin)
# Zero-crossing count of Δt(Q) curve
# CC/CV phase tagging via Q_CC_max from each run

import numpy as np
import pandas as pd

def t_at_Q_first_passage(Q_target, Q_traj, t_traj):
    Q = np.asarray(Q_traj); t = np.asarray(t_traj)
    above = Q >= Q_target
    if not above.any(): return None
    idx = np.argmax(above)
    if idx == 0: return float(t[0])
    return float(t[idx-1] + (Q_target - Q[idx-1]) / (Q[idx] - Q[idx-1]) * (t[idx] - t[idx-1]))


def compute_curve_metrics(res_DCAC, res_DC, Q_lo=1025, Q_hi=4101, n_grid=80, safety_margin=50):
    """A_Δt over Q-window in CC phase. Returns None if window invalid."""
    if res_DCAC.get('status') != 'ok' or res_DC.get('status') != 'ok':
        return None
    
    Q_DCAC = np.asarray(res_DCAC['Q_net_trajectory'])
    t_DCAC = np.asarray(res_DCAC['t_chg'])
    Q_DC   = np.asarray(res_DC['Q_net_trajectory'])
    t_DC   = np.asarray(res_DC['t_chg'])
    
    cc_DCAC_s = res_DCAC['CC_time_min'] * 60.0
    cc_DC_s   = res_DC['CC_time_min']   * 60.0
    
    Q_CC_max_DCAC = np.interp(cc_DCAC_s, t_DCAC, Q_DCAC)
    Q_CC_max_DC   = np.interp(cc_DC_s,   t_DC,   Q_DC)
    Q_CC_min      = min(Q_CC_max_DCAC, Q_CC_max_DC)
    
    Q_hi_eff = min(Q_hi, Q_CC_min - safety_margin)
    if Q_hi_eff <= Q_lo + 100:
        return None
    
    Q_grid = np.linspace(Q_lo, Q_hi_eff, n_grid)
    t_DCAC_at_Q = np.array([t_at_Q_first_passage(Q, Q_DCAC, t_DCAC) for Q in Q_grid])
    t_DC_at_Q   = np.array([t_at_Q_first_passage(Q, Q_DC,   t_DC)   for Q in Q_grid])
    t_DCAC_at_Q = np.array([x if x is not None else np.nan for x in t_DCAC_at_Q])
    t_DC_at_Q   = np.array([x if x is not None else np.nan for x in t_DC_at_Q])
    
    valid = ~(np.isnan(t_DCAC_at_Q) | np.isnan(t_DC_at_Q))
    if valid.sum() < 5:
        return None
    
    Q_v  = Q_grid[valid]
    dt_v = (t_DC_at_Q[valid] - t_DCAC_at_Q[valid]) / 60.0
    
    try:
        A_dt = float(np.trapezoid(dt_v, Q_v))
    except AttributeError:
        A_dt = float(np.trapz(dt_v, Q_v))
    
    s = np.sign(dt_v)
    s[s == 0] = 1
    n_cross = int(np.sum(np.abs(np.diff(s)) > 0))
    
    Q_width = Q_v[-1] - Q_v[0]
    avg_dt = A_dt / Q_width if Q_width > 0 else 0
    
    return {
        'Q_window': (float(Q_v[0]), float(Q_v[-1])),
        'A_dt': A_dt, 'avg_dt': avg_dt,
        'n_cross': n_cross, 'range_dt': float(dt_v.max() - dt_v.min()),
        'Q_CC_max_DCAC': float(Q_CC_max_DCAC), 'Q_CC_max_DC': float(Q_CC_max_DC),
    }


# === Run ===
print("=" * 110)
print("Curve metrics on 24 cases × 2 configs")
print("Q-window: Q20=1025 mAh → min(Q80=4101, Q_CC_max - 50 mAh) — CC phase only")
print("A_Δt = ∫Δt(Q) dQ; avg_dt = A_Δt / Q_width [minutes]; n_cross = sign-flip count of Δt(Q)")
print("=" * 110)

curve_summary = []
for i, row in df_cases.iterrows():
    DC, AC, f_Hz = row['DC'], row['AC'], row['f_Hz']
    cond = row['condition']
    
    res_X5A = run_single_case_X5A_noplating(DC, AC, f_Hz)
    res_pl  = run_single_case_plating(I_DC_Crate=DC, A_Crate=AC, f_Hz=f_Hz)
    
    m_X5A = compute_curve_metrics(res_X5A, dc_X5A[DC])
    m_pl  = compute_curve_metrics(res_pl,  dc_pl[DC])
    
    rec = {'condition': cond, 'kappa': row['kappa'], 'tau_label': row['tau_label'],
           'DC': DC, 'AC': AC, 'f_Hz': f_Hz}
    
    if m_X5A is not None:
        rec.update({
            'X5A_avg_dt': m_X5A['avg_dt'], 'X5A_A_dt': m_X5A['A_dt'],
            'X5A_n_cross': m_X5A['n_cross'], 'X5A_range': m_X5A['range_dt'],
            'X5A_CC_max_dcac': m_X5A['Q_CC_max_DCAC'], 'X5A_CC_max_dc': m_X5A['Q_CC_max_DC'],
            'X5A_window_hi': m_X5A['Q_window'][1],
        })
    if m_pl is not None:
        rec.update({
            'pl_avg_dt': m_pl['avg_dt'], 'pl_A_dt': m_pl['A_dt'],
            'pl_n_cross': m_pl['n_cross'], 'pl_range': m_pl['range_dt'],
            'pl_CC_max_dcac': m_pl['Q_CC_max_DCAC'], 'pl_CC_max_dc': m_pl['Q_CC_max_DC'],
            'pl_window_hi': m_pl['Q_window'][1],
        })
    
    if m_X5A is not None and m_pl is not None:
        rec['avg_dt_diff_pl_minus_X5A'] = m_pl['avg_dt'] - m_X5A['avg_dt']
        rec['meaningful_flip'] = (
            np.sign(m_X5A['avg_dt']) != np.sign(m_pl['avg_dt'])
            and abs(m_X5A['avg_dt']) > 0.1 and abs(m_pl['avg_dt']) > 0.1
        )
    else:
        rec['avg_dt_diff_pl_minus_X5A'] = None
        rec['meaningful_flip'] = False
    
    curve_summary.append(rec)

df_curve = pd.DataFrame(curve_summary)

# Print compact view
print(f"\n{'condition':<22} {'κ':>5} {'τ':<7} | {'X5A avg_dt':>11} {'cross':>5} | "
      f"{'pl avg_dt':>10} {'cross':>5} | {'pl-X5A':>8} {'flip':>5}")
print("-" * 110)
for _, r in df_curve.iterrows():
    a1 = f"{r.get('X5A_avg_dt', np.nan):>+11.4f}" if pd.notna(r.get('X5A_avg_dt')) else f"{'N/A':>11}"
    a2 = f"{r.get('pl_avg_dt',  np.nan):>+10.4f}"  if pd.notna(r.get('pl_avg_dt'))  else f"{'N/A':>10}"
    c1 = f"{r.get('X5A_n_cross', -1):>5d}" if pd.notna(r.get('X5A_n_cross')) else f"{'N/A':>5}"
    c2 = f"{r.get('pl_n_cross',  -1):>5d}" if pd.notna(r.get('pl_n_cross'))  else f"{'N/A':>5}"
    diff = f"{r['avg_dt_diff_pl_minus_X5A']:>+8.3f}" if pd.notna(r['avg_dt_diff_pl_minus_X5A']) else f"{'N/A':>8}"
    flip = "⚠" if r['meaningful_flip'] else "✓"
    print(f"{r['condition']:<22} {r['kappa']:>5.2f} {r['tau_label']:<7} | {a1} {c1} | {a2} {c2} | {diff} {flip:>5}")

# === Sign-coincidence under curve method ===
print("\n" + "=" * 80)
print("Curve method sign analysis")
print("=" * 80)

def sgn_thresh(x, thr=0.10):
    if pd.isna(x): return None
    if abs(x) < thr: return 0
    return 1 if x > 0 else -1

df_merged = df_curve.merge(df_cases[['condition', 'dt_exp']], on='condition', how='left')
s_X5A = df_merged['X5A_avg_dt'].apply(sgn_thresh)
s_pl  = df_merged['pl_avg_dt'].apply(sgn_thresh)
s_exp = df_merged['dt_exp'].apply(sgn_thresh)

agree_X5A_pl  = ((s_X5A == s_pl) & s_X5A.notna() & s_pl.notna()).sum()
agree_X5A_exp = ((s_X5A == s_exp) & s_X5A.notna() & s_exp.notna()).sum()
agree_pl_exp  = ((s_pl  == s_exp) & s_pl.notna() & s_exp.notna()).sum()
n_total = len(df_merged)
n_flip  = df_curve['meaningful_flip'].sum()

print(f"Total cases: {n_total}")
print(f"\nDay 11 vs X5-A (sim-vs-sim, curve method):")
print(f"  Sign agreement:     {agree_X5A_pl}/{n_total}")
print(f"  Meaningful flips:   {n_flip}/{n_total}")
print(f"  (Q80 scalar method previously reported: 1/24 flips)")
print(f"\nDescriptive vs MJ1 exp Δt(Q80):")
print(f"  X5-A curve method:   {agree_X5A_exp}/{n_total}  (compare: 15/24 from Q80 scalar)")
print(f"  Day 11 plating:      {agree_pl_exp}/{n_total}")

# Save
df_curve.to_csv(repo / "data" / "results_day11_curve_metrics.csv", index=False)
print(f"\n✓ Saved: data/results_day11_curve_metrics.csv")

Curve metrics on 24 cases × 2 configs
Q-window: Q20=1025 mAh → min(Q80=4101, Q_CC_max - 50 mAh) — CC phase only
A_Δt = ∫Δt(Q) dQ; avg_dt = A_Δt / Q_width [minutes]; n_cross = sign-flip count of Δt(Q)

condition                  κ τ       |  X5A avg_dt cross |  pl avg_dt cross |   pl-X5A  flip
--------------------------------------------------------------------------------------------------------------
0.1+0.9C 1τ             9.00 1τ      |     -1.0831     0 |    -0.9992     0 |   +0.084     ✓
0.1+0.2C 1τ             2.00 1τ      |     -0.3537     0 |    -0.3582     0 |   -0.004     ✓
0.2+0.3C 1τ             1.50 1τ      |     -0.2837     0 |    -0.2952     0 |   -0.011     ✓
0.2+0.3C 10τ            1.50 10τ     |     -2.8574     0 |    -2.8508     0 |   +0.007     ✓
0.2+0.3C 34.8τ          1.50 34.8τ   |     -8.3202     0 |    -8.2928     0 |   +0.027     ✓
0.2+0.8C 0.1τ           4.00 0.1τ    |     -0.3716     0 |    -0.3909     0 |   -0.019     ✓
0.2+0.8C 1τ             4.00 1τ      

In [21]:
# Stress test: A_Δt with multiple Q-windows
print("=" * 100)
print("A_Δt sensitivity to Q-window choice — X5-A no-plating, 6 representative cases")
print("=" * 100)

windows = {
    "Q20-Q80 (full CC)":      (1025, 4101),
    "Q20-Q60 (CC early/mid)": (1025, 3076),
    "Q40-Q70 (CC mid)":       (2050, 3588),
    "Q60-Q90 (CC late)":      (3076, 4613),
}

# Pick 6 cases spanning κ and τ
test_cases = [
    (0.1, 0.9, 0.01430, "0.1+0.9C 1τ", "κ=9, 1τ"),
    (0.2, 0.3, 0.00143, "0.2+0.3C 10τ", "κ=1.5, 10τ"),
    (0.2, 0.8, 0.00143, "0.2+0.8C 10τ", "κ=4, 10τ"),
    (0.3, 0.7, 0.00143, "0.3+0.7C 10τ", "κ=2.33, 10τ"),
    (0.4, 0.6, 0.00143, "0.4+0.6C 10τ", "κ=1.5, 10τ"),
    (0.9, 0.1, 0.01430, "0.9+0.1C 1τ", "κ=0.11, 1τ"),
]

# Header
print(f"{'case':<22} {'profile':<14} | ", end="")
for wlabel in windows.keys():
    print(f"{wlabel:>22}", end="")
print(f"  {'Q80 scalar':>12}")
print("-" * 130)

# Pre-fetch from earlier runs (cheap to recompute too)
for DC, AC, f_Hz, label, profile in test_cases:
    res_X5A = run_single_case_X5A_noplating(DC, AC, f_Hz)
    
    row_str = f"{label:<22} {profile:<14} | "
    for wlabel, (Q_lo, Q_hi) in windows.items():
        m = compute_curve_metrics(res_X5A, dc_X5A[DC], Q_lo=Q_lo, Q_hi=Q_hi, n_grid=80, safety_margin=20)
        avg = m['avg_dt'] if m else None
        row_str += f"{avg:>+22.4f}" if avg is not None else f"{'N/A':>22}"
    
    # Q80 single-point comparison
    Q_DCAC = np.asarray(res_X5A['Q_net_trajectory'])
    t_DCAC = np.asarray(res_X5A['t_chg'])
    Q_DC   = np.asarray(dc_X5A[DC]['Q_net_trajectory'])
    t_DC   = np.asarray(dc_X5A[DC]['t_chg'])
    t80_DCAC = t_at_Q_first_passage(4101, Q_DCAC, t_DCAC)
    t80_DC   = t_at_Q_first_passage(4101, Q_DC,   t_DC)
    dt80 = (t80_DC - t80_DCAC) / 60 if (t80_DC and t80_DCAC) else None
    row_str += f"  {dt80:>+12.3f}" if dt80 is not None else f"  {'N/A':>12}"
    
    print(row_str)

A_Δt sensitivity to Q-window choice — X5-A no-plating, 6 representative cases
case                   profile        |      Q20-Q80 (full CC)Q20-Q60 (CC early/mid)      Q40-Q70 (CC mid)     Q60-Q90 (CC late)    Q80 scalar
----------------------------------------------------------------------------------------------------------------------------------
0.1+0.9C 1τ            κ=9, 1τ        |                -1.0894               -0.9595               -1.0573               -1.3525        +2.178
0.2+0.3C 10τ           κ=1.5, 10τ     |                -2.8574               -2.5366               -2.7819               -2.5846        -5.297
0.2+0.8C 10τ           κ=4, 10τ       |                -3.6293               -3.7792               -3.8243               -3.6207        +9.326
0.3+0.7C 10τ           κ=2.33, 10τ    |                -3.1440               -3.1476               -3.4871               -3.1993        +7.269
0.4+0.6C 10τ           κ=1.5, 10τ     |                -2.5913              

In [22]:
# Check what MJ1 experimental data we actually have
import pandas as pd
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent

print("=" * 70)
print("Inventory: MJ1 experimental data files")
print("=" * 70)

candidates = [
    "figure1_master_table_cleaned.csv",
    "delta_tq_curve_data_strict_net.csv",
    "0.3C dc.csv",
]

for fname in candidates:
    fp = repo / "data" / fname
    if not fp.exists():
        print(f"\n{fname}: NOT FOUND")
        continue
    df = pd.read_csv(fp)
    print(f"\n=== {fname} ===")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print(df.head(2).to_string())

Inventory: MJ1 experimental data files

=== figure1_master_table_cleaned.csv ===
shape: (30, 28)
columns: ['DC_base', 'Condition', 'f_Hz', 'DC_C', 'AC_C', 'I_DC_A', 'A_A', 'kappa', 'tau_ref_s', 'Pi', 'log10Π', 'Delta_t_Q80_min', 'Regime', 'total_time_min', 'baseline_total_min', 'Delta_T_total_min', 'kappa_old', 'Pi_old', 'Pi_diff', 'log10Π_old', 'log10Π_diff', 'kappa_recomputed', 'Unnamed: 22', 'T_max', 'T_avg', 'U00_12h(V)', 'CC_min', 'CV_min']
  DC_base    Condition    f_Hz  DC_C  AC_C  I_DC_A   A_A  kappa  tau_ref_s        Pi    log10Π  Delta_t_Q80_min      Regime  total_time_min  baseline_total_min  Delta_T_total_min  kappa_old    Pi_old       Pi_diff  log10Π_old   log10Π_diff  kappa_recomputed  Unnamed: 22   T_max   T_avg  U00_12h(V)      CC_min  CV_min
0    0.1C  0.1+0.9C 1τ  0.0143   0.1   0.9    0.34  3.06    9.0  20.289285  0.290137 -0.537397             2.88  Misleading      523.416667          593.083333          69.666667        9.0  0.290137 -4.610000e-10   -0.537397 -6.90

In [23]:
# MJ1 experimental A_Δt curve method vs sim
# Use delta_tq_curve_data_strict_net.csv to compute MJ1 A_Δt and compare to sim

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent

# === Load MJ1 curve data ===
df_mj1 = pd.read_csv(repo / "data" / "delta_tq_curve_data_strict_net.csv")
print(f"MJ1 curve data: {df_mj1.shape}")
print(f"Unique protocols: {df_mj1['protocol'].nunique()}")
print(f"Unique reference_protocols: {df_mj1['reference_protocol'].nunique()}")
print(f"\nFirst 3 protocol entries:")
print(df_mj1[['protocol', 'reference_protocol']].drop_duplicates().head(10).to_string())

# === Map MJ1 protocol filenames to our condition labels ===
# MJ1 protocol naming: e.g. "0.3+0.7 0.1tau.csv" → our "0.3+0.7C 0.1τ"
# Need to construct mapping

mj1_protocols = df_mj1['protocol'].unique()
print(f"\nAll {len(mj1_protocols)} MJ1 protocols:")
for p in sorted(mj1_protocols):
    print(f"  {p}")

MJ1 curve data: (6000, 6)
Unique protocols: 3
Unique reference_protocols: 1

First 3 protocol entries:
                protocol reference_protocol
0     0.3+0.7 0.1tau.csv        0.3C dc.csv
2000    0.3+0.7 1tau.csv        0.3C dc.csv
4000  0.3+0.7 10 tau.csv        0.3C dc.csv

All 3 MJ1 protocols:
  0.3+0.7 0.1tau.csv
  0.3+0.7 10 tau.csv
  0.3+0.7 1tau.csv


In [24]:
# MJ1 vs sim A_Δt cross-method comparison on the 3 protocols with curve data
import numpy as np
import pandas as pd
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent

df_mj1 = pd.read_csv(repo / "data" / "delta_tq_curve_data_strict_net.csv")

# MJ1 raw data is in Ah; sim is in mAh. Q-window: 1.025 Ah → 4.101 Ah (= Q20 to Q80 in mAh terms)
Q_LO_AH = 1.025
Q_HI_AH = 4.101

# Map MJ1 protocol filenames to (DC, AC, f_Hz, condition)
mj1_to_case = {
    "0.3+0.7 0.1tau.csv": (0.3, 0.7, 0.14300, "0.3+0.7C 0.1τ"),
    "0.3+0.7 1tau.csv":   (0.3, 0.7, 0.01430, "0.3+0.7C 1τ"),
    "0.3+0.7 10 tau.csv": (0.3, 0.7, 0.00143, "0.3+0.7C 10τ"),
}

print("=" * 110)
print("MJ1 (exp) vs X5-A (sim) A_Δt curve comparison — 3 cases with MJ1 curve data")
print("=" * 110)
print(f"Q-window: {Q_LO_AH}-{Q_HI_AH} Ah (= Q20-Q80 mAh equivalent)")
print()

print(f"{'condition':<22} | {'MJ1 Q80 scalar':>16} {'MJ1 A_Δt avg':>16} {'MJ1 sign':>10} | "
      f"{'sim X5-A A_Δt avg':>20} {'sim sign':>10} | {'sim/exp coincide?':>20}")
print("-" * 130)

results = []
for proto, (DC, AC, f_Hz, cond) in mj1_to_case.items():
    # === MJ1 curve ===
    mask = df_mj1['protocol'] == proto
    Q_arr = df_mj1[mask]['Q_net_Ah'].values
    dt_arr = df_mj1[mask]['Delta_t_Q_min'].values
    
    in_window = (Q_arr >= Q_LO_AH) & (Q_arr <= Q_HI_AH)
    Q_w = Q_arr[in_window]
    dt_w = dt_arr[in_window]
    
    if len(Q_w) < 5:
        print(f"{cond:<22}: insufficient MJ1 curve data in window (n={len(Q_w)})")
        continue
    
    sort_idx = np.argsort(Q_w)
    Q_w = Q_w[sort_idx]
    dt_w = dt_w[sort_idx]
    
    try:
        A_dt_mj1 = np.trapezoid(dt_w, Q_w)
    except AttributeError:
        A_dt_mj1 = np.trapz(dt_w, Q_w)
    
    Q_width = Q_w[-1] - Q_w[0]
    avg_dt_mj1 = A_dt_mj1 / Q_width if Q_width > 0 else 0
    
    # MJ1 Q80 scalar
    mj1_Q80_scalar = float(df_4way[df_4way['condition'] == cond]['dt_exp'].iloc[0])
    
    # === sim X5-A curve (re-run since we have it) ===
    res_sim = run_single_case_X5A_noplating(DC, AC, f_Hz)
    m_sim = compute_curve_metrics(res_sim, dc_X5A[DC], 
                                    Q_lo=Q_LO_AH*1000, Q_hi=Q_HI_AH*1000, 
                                    n_grid=80, safety_margin=20)
    sim_avg_dt = m_sim['avg_dt'] if m_sim else None
    
    # Signs
    sgn = lambda x, thr=0.10: 0 if (x is None or abs(x) < thr) else (1 if x > 0 else -1)
    s_mj1 = sgn(avg_dt_mj1)
    s_sim = sgn(sim_avg_dt)
    coincide = "✓ YES" if s_mj1 == s_sim else "✗ NO"
    
    sgn_str_mj1 = '+' if s_mj1 == 1 else ('-' if s_mj1 == -1 else '0')
    sgn_str_sim = '+' if s_sim == 1 else ('-' if s_sim == -1 else '0')
    
    print(f"{cond:<22} | {mj1_Q80_scalar:>+16.3f} {avg_dt_mj1:>+16.4f} {sgn_str_mj1:>10} | "
          f"{sim_avg_dt:>+20.4f} {sgn_str_sim:>10} | {coincide:>20}")
    
    results.append({'condition': cond, 'mj1_Q80': mj1_Q80_scalar, 'mj1_A_dt_avg': avg_dt_mj1,
                    's_mj1': s_mj1, 'sim_A_dt_avg': sim_avg_dt, 's_sim': s_sim, 'coincide': s_mj1 == s_sim})

print()
df_cross = pd.DataFrame(results)
n_coincide = df_cross['coincide'].sum()
print(f"Curve method sim/exp sign-coincidence: {n_coincide}/3")

MJ1 (exp) vs X5-A (sim) A_Δt curve comparison — 3 cases with MJ1 curve data
Q-window: 1.025-4.101 Ah (= Q20-Q80 mAh equivalent)

condition              |   MJ1 Q80 scalar     MJ1 A_Δt avg   MJ1 sign |    sim X5-A A_Δt avg   sim sign |    sim/exp coincide?
----------------------------------------------------------------------------------------------------------------------------------
0.3+0.7C 0.1τ          |           +0.020          +1.8235          + |              -0.2351          - |                 ✗ NO
0.3+0.7C 1τ            |           +2.970          +2.5856          + |              -0.4273          - |                 ✗ NO
0.3+0.7C 10τ           |           +8.670          +8.4406          + |              -3.1440          - |                 ✗ NO

Curve method sim/exp sign-coincidence: 0/3
